In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

In [2]:
def get_text_cols(df):
    return [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]

print("pandas version:", pd.__version__)

pandas version: 3.0.6


In [3]:
RAW_DIR   = Path("../data/raw")
CLEAN_DIR = Path("../data/clean")

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("See the following files in the raw data directory:")
for f in sorted(RAW_DIR.glob("*.csv")):
    print("  -", f.name)

See the following files in the raw data directory:
  - Customer_Service.csv
  - Customers.csv
  - Group_Operations.csv
  - Inventory_History.csv
  - Locations.csv
  - Product_Master.csv
  - Production_Orders.csv
  - Sales_Transactions.csv


In [4]:
# ============================================================
# 0.3 Load the 7 reference files (Arabic-safe)
# ============================================================
def read_csv_safe(path):
    """
    Read a CSV trying several encodings, so Arabic text is read correctly.
    - utf-8-sig : UTF-8 with or without BOM (Excel adds a hidden BOM) -> try first
    - cp1256    : old Windows Arabic encoding -> fallback
    Returns the DataFrame and the encoding that worked.
    """
    for enc in ["utf-8-sig", "cp1256"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            # Clean column names: remove hidden BOM characters and extra spaces
            df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()
            return df, enc
        except UnicodeDecodeError:
            continue   # this encoding failed -> try the next one
    raise ValueError(f"Could not read {path} with any known encoding")


# Dictionary: short name (key) -> real file name (value)
files = {
    "products":         "Product_Master.csv",
    "locations":        "Locations.csv",
    "customers":        "Customers.csv",
    "inventory":        "Inventory_History.csv",
    "production":       "Production_Orders.csv",
    "customer_service": "Customer_Service.csv",
    "group_ops":        "Group_Operations.csv",
}

raw = {}        # will hold the loaded DataFrames
encodings = {}  # will record which encoding worked for each file

for name, filename in files.items():
    raw[name], encodings[name] = read_csv_safe(RAW_DIR / filename)
    print(f"Loaded {name:<18} {raw[name].shape[0]:>8,} rows   "
          f"{raw[name].shape[1]:>3} cols   encoding={encodings[name]}")

Loaded products                 59 rows    12 cols   encoding=utf-8-sig
Loaded locations                 5 rows     9 cols   encoding=utf-8-sig
Loaded customers            30,000 rows     9 cols   encoding=utf-8-sig
Loaded inventory             6,372 rows    19 cols   encoding=utf-8-sig
Loaded production            2,500 rows    16 cols   encoding=utf-8-sig
Loaded customer_service      6,500 rows     9 cols   encoding=utf-8-sig
Loaded group_ops               180 rows     7 cols   encoding=utf-8-sig


In [5]:
# ============================================================
# 0.3b Verify Arabic text was read correctly
# ============================================================
# NOTE: no "r" prefix here on purpose.
# Python converts \u0600 into the real Arabic character BEFORE passing it
# to the regex engine (pandas 3 uses pyarrow's RE2 engine, which does not
# understand the "\u0600" escape inside a raw string).
arabic_pattern  = "[\u0600-\u06FF]"   # any Arabic letter
garbled_pattern = "[ØÙÚ]"             # signs of Arabic read with the wrong encoding

for name, df in raw.items():
    for col in get_text_cols(df):
        values = df[col].dropna().astype(str)
        has_arabic  = values.str.contains(arabic_pattern,  regex=True)
        has_garbled = values.str.contains(garbled_pattern, regex=True)

        if has_arabic.any():
            example = values[has_arabic].iloc[0]
            print(f"[Arabic OK] {name}.{col}: {has_arabic.sum()} rows  e.g. -> {example}")
        if has_garbled.any():
            example = values[has_garbled].iloc[0]
            print(f"[GARBLED!] {name}.{col}: {has_garbled.sum()} rows  e.g. -> {example}")

[Arabic OK] products.Product_Name: 14 rows  e.g. -> Mustard - طقم تيشيرت وشورت اطفال صيفي
[Arabic OK] inventory.Product_Name: 1512 rows  e.g. -> Mustard - طقم تيشيرت وشورت اطفال صيفي
[Arabic OK] production.Product_Name: 600 rows  e.g. -> Scooby-Doo - بيجامة حريمي صيفي2024


Product_Master

In [6]:

# 1.1 Working copy + first look
prod = raw["products"].copy()
print("Shape (rows, cols):", prod.shape)
prod.head(10) 

Shape (rows, cols): (59, 12)


,Product_ID,SKU,Product_Name,Brand,Product_Family,Gender,Audience,Catalog_Season,Sizes,Public_Retail_Price_EGP,Public_Catalog_Source,Verification_Class
0,P001,26115-8,Summer children's girls-Happy,Leo Moda,Pajamas/Homewear,Girls,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
1,P002,26115-6,Summer children's girls-turtle,Leo Moda,Pajamas/Homewear,Girls,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
2,P003,26115-5,Summer children's girls-Hello kitty Rose & turquoise,Leo Moda,Pajamas/Homewear,Girls,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
3,P004,26115-7,Summer children's girls-Hello kitty black&white,Leo Moda,Pajamas/Homewear,Girls,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
4,P005,26115-2,Summer children's boys-Black and orange,Leo Moda,Pajamas/Homewear,Boys,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
5,P006,26115-1,Summer children's boys-Sonic,Leo Moda,Pajamas/Homewear,Boys,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
6,P007,26115-3,Summer children's boys-Black and turquoise,Leo Moda,Pajamas/Homewear,Boys,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
7,P008,26115-4,Summer children's boys-SpongeBob,Leo Moda,Pajamas/Homewear,Boys,Kids,Summer 2026,"4, 6, 8, 10, 12",348,https://leo.moda/shop/,Public catalog product / public retail price
8,P009,LEO-WC-P-5404,Boys' summer t-shirt-Aubergine,Leo Moda,T-Shirts,Boys,Kids,Summer 2026,"4, 6, 8, 10, 12",330,https://leo.moda/shop/,Public catalog product / public retail price
9,P010,LEO-WC-P-5403,Boys' summer t-shirt-olive,Leo Moda,T-Shirts,Boys,Kids,Summer 2026,"4, 6, 8, 10, 12",330,https://leo.moda/shop/,Public catalog product / public retail price


In [7]:
# ============================================================
# 1.2 Data types & missing values
# ============================================================
prod.info()
print("\nMissing values per column:")
print(prod.isna().sum())

# What to check:
# - Product_ID must have 0 missing (it's the key)
# - Public_Retail_Price_EGP must be numeric (int64/float64), NOT object

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Product_ID               59 non-null     str  
 1   SKU                      59 non-null     str  
 2   Product_Name             59 non-null     str  
 3   Brand                    59 non-null     str  
 4   Product_Family           59 non-null     str  
 5   Gender                   59 non-null     str  
 6   Audience                 59 non-null     str  
 7   Catalog_Season           59 non-null     str  
 8   Sizes                    59 non-null     str  
 9   Public_Retail_Price_EGP  59 non-null     int64
 10  Public_Catalog_Source    59 non-null     str  
 11  Verification_Class       59 non-null     str  
dtypes: int64(1), str(11)
memory usage: 15.4 KB

Missing values per column:
Product_ID                 0
SKU                        0
Product_Name               0
Brand            

In [8]:
# ============================================================
# 1.3 Duplicates

print("Duplicated Product_ID:", prod["Product_ID"].duplicated().sum())
print("Duplicated SKU:       ", prod["SKU"].duplicated().sum())
print("Fully duplicated rows:", prod.duplicated().sum())


Duplicated Product_ID: 0
Duplicated SKU:        0
Fully duplicated rows: 0


In [9]:
# ============================================================
# 1.4 Distinct values in categorical columns

category_cols = ["Brand", "Product_Family", "Gender", "Audience", "Catalog_Season", "Sizes"]

for col in category_cols:
    # nunique() -> number of distinct values
    print(f"\n========== {col} ({prod[col].nunique()} distinct values) ==========")
    print(prod[col].value_counts(dropna=False))

# What to look for:
# - Same value written differently (e.g. "Girls" vs "girls" vs "Girl")
# - Strange values (e.g. "N/A", "-")
# - Season formats ("Summer" vs "Summer 2024")


========== Brand (1 distinct values) ==========
Brand
Leo Moda    59
Name: count, dtype: int64

========== Product_Family (4 distinct values) ==========
Product_Family
Pajamas/Homewear    42
T-Shirts             6
Sweatshirts          6
Kids Sets            5
Name: count, dtype: int64

========== Gender (4 distinct values) ==========
Gender
Boys      25
Girls     21
Women      9
Unisex     4
Name: count, dtype: int64

========== Audience (2 distinct values) ==========
Audience
Kids     50
Women     9
Name: count, dtype: int64

========== Catalog_Season (5 distinct values) ==========
Catalog_Season
Summer 2026      31
Summer 2024       9
Autumn 2025       8
Autumn/Winter     6
Summer            5
Name: count, dtype: int64

========== Sizes (2 distinct values) ==========
Sizes
4, 6, 8, 10, 12    50
M, L, XL, 2XL       9
Name: count, dtype: int64


In [10]:
# ============================================================
# 1.5 Strip leading/trailing spaces from all text columns
# ============================================================
# select_dtypes(include="object") -> text columns only
text_cols = prod.select_dtypes(include="object").columns

for col in text_cols:
    prod[col] = prod[col].str.strip()

print("Stripped columns:", list(text_cols))


Stripped columns: ['Product_ID', 'SKU', 'Product_Name', 'Brand', 'Product_Family', 'Gender', 'Audience', 'Catalog_Season', 'Sizes', 'Public_Catalog_Source', 'Verification_Class']


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16588\2560557153.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = prod.select_dtypes(include="object").columns


In [11]:
# ============================================================
# 1.5b Tidy Product_Name (display fix only — the name itself is NOT changed)
# ============================================================

ARABIC = "\u0600-\u06FF" 

def tidy_name(text):
    if pd.isna(text):
        return text
    s = str(text)
    s = re.sub(f"([{ARABIC}])(\\d)", r"\1 \2", s)   # Arabic letter then digit -> add space
    s = re.sub(f"(\\d)([{ARABIC}])", r"\1 \2", s)   # digit then Arabic letter -> add space
    s = re.sub(r"\s+", " ", s).strip()             # collapse multiple spaces into one
    return s

before = prod["Product_Name"].copy()
prod["Product_Name"] = prod["Product_Name"].apply(tidy_name)

# Show only the names that actually changed
changed = before != prod["Product_Name"]
print(f"Names changed: {changed.sum()} of {len(prod)}")
pd.DataFrame({"before": before[changed], "after": prod.loc[changed, "Product_Name"]})

Names changed: 7 of 59


,before,after
50,Ice cream - بيجامة حريمي صيفي2024,Ice cream - بيجامة حريمي صيفي 2024
51,Teddy bear - بيجامة حريمي صيفي2024,Teddy bear - بيجامة حريمي صيفي 2024
52,PINK panther - بيجامة حريمي صيفي2024,PINK panther - بيجامة حريمي صيفي 2024
53,Gazelle - بيجامة حريمي صيفي2024,Gazelle - بيجامة حريمي صيفي 2024
54,Scooby-Doo - بيجامة حريمي صيفي2024,Scooby-Doo - بيجامة حريمي صيفي 2024
55,BATGIRL - بيجامة حريمي صيفي2024,BATGIRL - بيجامة حريمي صيفي 2024
58,Positive vibes - 2024بيجامة حريمي صيفي,Positive vibes - 2024 بيجامة حريمي صيفي


In [12]:
# ============================================================
# 1.6 Drop columns not needed for analysis
# ============================================================
# Public_Catalog_Source -> website link (reference only)
# Verification_Class    -> constant label ("public catalog product")
cols_to_drop = ["Public_Catalog_Source", "Verification_Class"]

prod = prod.drop(columns=cols_to_drop)

print("Remaining columns:", list(prod.columns))

Remaining columns: ['Product_ID', 'SKU', 'Product_Name', 'Brand', 'Product_Family', 'Gender', 'Audience', 'Catalog_Season', 'Sizes', 'Public_Retail_Price_EGP']


In [13]:
# ============================================================
# 1.7 Sizes: text -> list + size type
# ============================================================
def parse_sizes(text):
    """Convert '4, 6, 8, 10, 12' into ['4', '6', '8', '10', '12']."""
    if pd.isna(text):                 # empty cell -> empty list (avoids errors)
        return []
    parts = str(text).split(",")      # split at every comma
    # For each part: strip spaces, uppercase ("xl" -> "XL"), skip empty parts
    return [s.strip().upper() for s in parts if s.strip() != ""]


def get_size_type(sizes):
    """Numeric sizes = Kids (age-based); letter sizes = Adult (M, L, XL)."""
    if len(sizes) == 0:
        return "Unknown"
    if all(s.isdigit() for s in sizes):       # every size is digits only
        return "Kids"
    if all(not s.isdigit() for s in sizes):   # no size is digits only
        return "Adult"
    return "Mixed"                            # mix of both -> needs review


# New columns (we keep the original "Sizes" column for reference)
prod["Sizes_List"] = prod["Sizes"].apply(parse_sizes)        # list of sizes
prod["Size_Type"]  = prod["Sizes_List"].apply(get_size_type) # Kids / Adult / Mixed / Unknown
prod["N_Sizes"]    = prod["Sizes_List"].apply(len)           # number of sizes offered

print(prod["Size_Type"].value_counts())
prod[["Product_ID", "Sizes", "Sizes_List", "Size_Type", "N_Sizes"]].head(10)

Size_Type
Kids     50
Adult     9
Name: count, dtype: int64


,Product_ID,Sizes,Sizes_List,Size_Type,N_Sizes
0,P001,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
1,P002,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
2,P003,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
3,P004,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
4,P005,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
5,P006,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
6,P007,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
7,P008,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
8,P009,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5
9,P010,"4, 6, 8, 10, 12","[4, 6, 8, 10, 12]",Kids,5


In [14]:
# ============================================================
# 1.8 Catalog_Season -> Season + Season_Year
# ============================================================
# Fashion industry uses 2 main seasons per year:
#   "Summer"        = Spring + Summer      (light clothing)
#   "Autumn/Winter" = Autumn + Winter      (heavy clothing)
# So "Autumn/Winter", "Autumn 2025" -> "Autumn/Winter"
#    "Summer", "Summer 2024"        -> "Summer"
def parse_season(text):
    """Map any season text to 'Summer' or 'Autumn/Winter' and extract the year."""
    if pd.isna(text):
        return pd.Series([None, None])

    t = str(text).lower()   # lowercase so "Winter" and "winter" match the same

    # Check cold-season words FIRST, so "Autumn/Winter" is not cut to "Autumn"
    if any(word in t for word in ["autumn", "fall", "winter"]):
        season = "Autumn/Winter"
    elif any(word in t for word in ["summer", "spring"]):
        season = "Summer"
    else:
        season = None       # unknown season -> leave empty (don't guess)

    # Year: "20" followed by exactly 2 digits (2000–2099)
    year_match = re.search(r"(20\d{2})", t)
    year = int(year_match.group(1)) if year_match else None

    return pd.Series([season, year])


prod[["Season", "Season_Year"]] = prod["Catalog_Season"].apply(parse_season)

# "Int64" (capital I) = integer type that ALLOWS missing values
prod["Season_Year"] = prod["Season_Year"].astype("Int64")

# Verify: each original format once, next to its parsed result
print(prod[["Catalog_Season", "Season", "Season_Year"]].drop_duplicates().to_string(index=False))
print("\nProducts per season:")
print(prod["Season"].value_counts(dropna=False))

Catalog_Season        Season  Season_Year
   Summer 2026        Summer         2026
 Autumn/Winter Autumn/Winter         <NA>
   Autumn 2025 Autumn/Winter         2025
        Summer        Summer         <NA>
   Summer 2024        Summer         2024

Products per season:
Season
Summer           45
Autumn/Winter    14
Name: count, dtype: int64


In [15]:
# ============================================================
# 1.9 Price: ensure numeric and sensible
# ============================================================
price_col = "Public_Retail_Price_EGP"

# Force numeric; anything that isn't a number becomes NaN (safety step)
prod[price_col] = pd.to_numeric(prod[price_col], errors="coerce")

# "|" = OR. Each condition must be inside its own ( )
bad_price = prod[prod[price_col].isna() | (prod[price_col] <= 0)]
print("Products with missing / zero / negative price:", len(bad_price))

# describe() -> count, mean, std, min, quartiles, max
print("\nPrice summary:")
print(prod[price_col].describe().round(1))

# groupby -> split into groups, then aggregate each group
print("\nPrice by Audience & Product_Family:")
print(prod.groupby(["Audience", "Product_Family"])[price_col]
          .agg(["count", "min", "mean", "max"])
          .round(0))

Products with missing / zero / negative price: 0

Price summary:
count     59.0
mean     354.7
std       76.6
min      289.0
25%      299.0
50%      330.0
75%      388.0
max      549.0
Name: Public_Retail_Price_EGP, dtype: float64

Price by Audience & Product_Family:
                           count  min   mean  max
Audience Product_Family                          
Kids     Kids Sets             5  400  400.0  400
         Pajamas/Homewear     33  289  308.0  348
         Sweatshirts           6  549  549.0  549
         T-Shirts              6  330  330.0  330
Women    Pajamas/Homewear      9  388  388.0  388


In [16]:
# ============================================================
# 1.10 Consistency checks between columns (sanity checks)
# ============================================================
# crosstab -> counts for every combination of two columns
# Expected: numbers only on the "diagonal" (Kids-Kids, Adult-Women)
print("========== Size_Type x Audience ==========")
print(pd.crosstab(prod["Size_Type"], prod["Audience"]))

print("\n========== Gender x Audience ==========")
print(pd.crosstab(prod["Gender"], prod["Audience"]))

# "&" = AND, "|" = OR, "!=" = not equal
# Flag 3 kinds of contradictions:
#   1) Mixed / Unknown size types
#   2) Kid sizes but audience is not Kids
#   3) Adult sizes but audience is Kids
problems = prod[
    (prod["Size_Type"].isin(["Mixed", "Unknown"])) |
    ((prod["Size_Type"] == "Kids")  & (prod["Audience"] != "Kids")) |
    ((prod["Size_Type"] == "Adult") & (prod["Audience"] == "Kids"))
]
print(f"\n========== Products with contradictions: {len(problems)} ==========")
# Expected: empty table
problems[["Product_ID", "Product_Name", "Gender", "Audience", "Sizes", "Size_Type"]]

========== Size_Type x Audience ==========
Audience   Kids  Women
Size_Type             
Adult         0      9
Kids         50      0

========== Gender x Audience ==========


Audience  Kids  Women
Gender               
Boys        25      0
Girls       21      0
Unisex       4      0
Women        0      9

========== Products with contradictions: 0 ==========


,Product_ID,Product_Name,Gender,Audience,Sizes,Size_Type


In [17]:
# ============================================================
# 1.11 Final review before saving
# ============================================================
# Row count must stay the same (we didn't remove any product)
print("Shape: before", raw["products"].shape, "-> after", prod.shape)

print("\nFinal columns & types:")
for col in prod.columns:
    print(f"  {col:<25} {str(prod[col].dtype):<10}")

# Missing values that remain (Season_Year can be empty — that's expected)
remaining_nulls = prod.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nRemaining missing values:", dict(remaining_nulls) if len(remaining_nulls) else "None")

# size() -> number of rows (products) in each group
print("\nProduct count by Audience & Product_Family:")
print(prod.groupby(["Audience", "Product_Family"]).size().to_string())

Shape: before (59, 12) -> after (59, 15)

Final columns & types:
  Product_ID                str       
  SKU                       str       
  Product_Name              str       
  Brand                     str       
  Product_Family            str       
  Gender                    str       
  Audience                  str       
  Catalog_Season            str       
  Sizes                     str       
  Public_Retail_Price_EGP   int64     
  Sizes_List                object    
  Size_Type                 str       
  N_Sizes                   int64     
  Season                    str       
  Season_Year               Int64     

Remaining missing values: {'Season_Year': np.int64(11)}

Product count by Audience & Product_Family:
Audience  Product_Family  
Kids      Kids Sets            5
          Pajamas/Homewear    33
          Sweatshirts          6
          T-Shirts             6
Women     Pajamas/Homewear     9


In [18]:
# ============================================================
# 1.12b Drop redundant / helper columns
# ============================================================
# Size_Type      -> identical to Audience 
# N_Sizes        -> constant per Audience (Kids=5, Women=4); 
# Sizes          -> replaced by Sizes_List (same info, list format)
# Catalog_Season -> replaced by Season + Season_Year
redundant_cols = ["Size_Type", "N_Sizes", "Sizes", "Catalog_Season"]

# Brand: drop only if it has a single value (a constant column adds no information)
print("Brand values:", prod["Brand"].unique())
if prod["Brand"].nunique() == 1:
    redundant_cols.append("Brand")
    print("-> Brand is constant, dropping it")
else:
    print("-> Brand has multiple values, keeping it")

prod = prod.drop(columns=redundant_cols)
print("\nRemaining columns:", list(prod.columns))

Brand values: <ArrowStringArray>
['Leo Moda']
Length: 1, dtype: str
-> Brand is constant, dropping it

Remaining columns: ['Product_ID', 'SKU', 'Product_Name', 'Product_Family', 'Gender', 'Audience', 'Public_Retail_Price_EGP', 'Sizes_List', 'Season', 'Season_Year']


In [19]:

prod.tail(20)

,Product_ID,SKU,Product_Name,Product_Family,Gender,Audience,Public_Retail_Price_EGP,Sizes_List,Season,Season_Year
39,P040,CAT-040,Boys' Autumn Pajamas-Loony Tunes,Pajamas/Homewear,Boys,Kids,289,"[4, 6, 8, 10, 12]",Autumn/Winter,2025
40,P041,CAT-041,Boys' Autumn Pajamas-Jurassic Park,Pajamas/Homewear,Boys,Kids,289,"[4, 6, 8, 10, 12]",Autumn/Winter,2025
41,P042,CAT-042,Girls' autumn pajamas-Cheeky Charmins,Pajamas/Homewear,Girls,Kids,289,"[4, 6, 8, 10, 12]",Autumn/Winter,2025
42,P043,CAT-043,Girls' autumn pajamas-Snow White,Pajamas/Homewear,Girls,Kids,289,"[4, 6, 8, 10, 12]",Autumn/Winter,2025
43,P044,CAT-044,Girls' autumn pajamas-Ice Cream,Pajamas/Homewear,Girls,Kids,289,"[4, 6, 8, 10, 12]",Autumn/Winter,2025
44,P045,CAT-045,Girls' autumn pajamas-Stitch,Pajamas/Homewear,Girls,Kids,289,"[4, 6, 8, 10, 12]",Autumn/Winter,2025
45,P046,CAT-046,Mustard - طقم تيشيرت وشورت اطفال صيفي,Kids Sets,Unisex,Kids,400,"[4, 6, 8, 10, 12]",Summer,<NA>
46,P047,CAT-047,Mauve - طقم تيشيرت وشورت اطفال صيفي,Kids Sets,Unisex,Kids,400,"[4, 6, 8, 10, 12]",Summer,<NA>
47,P048,CAT-048,Semon - طقم تيشيرت وشورت اطفال صيفي,Kids Sets,Unisex,Kids,400,"[4, 6, 8, 10, 12]",Summer,<NA>
48,P049,CAT-049,Fuchsia - طقم تيشيرت وشورت اطفال صيفي,Kids Sets,Girls,Kids,400,"[4, 6, 8, 10, 12]",Summer,<NA>


In [20]:
# ============================================================
# 1.12 Save
# ============================================================
# Parquet -> used by the code (fast, small, keeps column types and lists)
prod.to_parquet(CLEAN_DIR / "products.parquet", index=False)

# CSV -> for manual review in Excel only
# CSV can't store lists, so convert Sizes_List back to text: ['4','6'] -> "4, 6"
prod_csv = prod.copy()
prod_csv["Sizes_List"] = prod_csv["Sizes_List"].apply(lambda x: ", ".join(x))
# utf-8-sig -> Arabic text displays correctly in Excel
prod_csv.to_csv(CLEAN_DIR / "products.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print(f"   {CLEAN_DIR / 'products.parquet'}   <- used by code")
print(f"   {CLEAN_DIR / 'products.csv'}       <- for Excel review")

Saved:
   ..\data\clean\products.parquet   <- used by code
   ..\data\clean\products.csv       <- for Excel review


###  Product_Master — Cleaning Summary
| Decision | Reason |
|---|---|
| Stripped spaces from all text columns | Filters and joins must match exactly |
| Dropped `Public_Catalog_Source`, `Verification_Class` | Reference only |
| `Sizes` → `Sizes_List` (list) | Used by the bot and the size planner |
| Verified `Audience` against sizes (0 contradictions) | Numeric sizes = Kids, letter sizes = Women |
| `Catalog_Season` → `Season` (Summer / Autumn/Winter) + `Season_Year` | 2-season industry standard; better for forecasting |
| `Season_Year` left empty when not stated (11 products) | No guessed values |
| Dropped `Size_Type`, `N_Sizes`, `Sizes`, `Catalog_Season` (+ `Brand` if constant) | Redundant after verification |

**Result:** 59 products — 50 Kids (Boys 25, Girls 21, Unisex 4), 9 Women (all pajamas)

Location 

In [21]:
# ============================================================
# 2.1 Working copy + first look
# ============================================================
loc = raw["locations"].copy()

print("Shape:", loc.shape)
loc   

Shape: (5, 9)


,Location_ID,Location_Name,Location_Type,Channel,Governorate,City,Public_Address,Source_URL,Verification_Class
0,LOC001,LEO MODA - Sidi Bishr Store,Retail Store,Retail,Alexandria,Sidi Bishr,"72C2+GXJ, Sidi Bishr Bahri, Montaza 2, Alexandria, Egypt",https://leo.moda/our-stores/,Public store / address cross-checked from public busines...
1,LOC002,LEO MODA Online Store,Digital Store,Online,Egypt,Nationwide,Online - delivery across Egypt,https://leo.moda/,Public channel
2,LOC003,El-Sharnouby Wholesale Desk,B2B / Wholesale,Wholesale,Alexandria,Zizinia,"5 Amin Yehia Street, Zizinia, Alexandria, Egypt",https://www.el-sharnouby.com/,Public HQ / wholesale inquiry channel
3,LOC004,El-Sharnouby Head Office,Head Office,Corporate,Alexandria,Zizinia,"5 Amin Yehia Street, Zizinia, Alexandria, Egypt",https://www.el-sharnouby.com/,Public HQ
4,LOC005,LEO Production Hub,Factory / Production,Operations,Alexandria,Alexandria,"Alexandria, Egypt",https://www.el-sharnouby.com/services,Public business unit; exact facility address not asserted


In [22]:
loc.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Location_ID         5 non-null      str  
 1   Location_Name       5 non-null      str  
 2   Location_Type       5 non-null      str  
 3   Channel             5 non-null      str  
 4   Governorate         5 non-null      str  
 5   City                5 non-null      str  
 6   Public_Address      5 non-null      str  
 7   Source_URL          5 non-null      str  
 8   Verification_Class  5 non-null      str  
dtypes: str(9)
memory usage: 1.3 KB


In [23]:
# ============================================================
# 2.3 Distinct values
# ============================================================
for col in ["Location_Type", "Channel", "Governorate", "City"]:
    print(f"\n========== {col} ==========")
    print(loc[col].value_counts(dropna=False))


========== Location_Type ==========
Location_Type
Retail Store            1
Digital Store           1
B2B / Wholesale         1
Head Office             1
Factory / Production    1
Name: count, dtype: int64

========== Channel ==========
Channel
Retail        1
Online        1
Wholesale     1
Corporate     1
Operations    1
Name: count, dtype: int64

========== Governorate ==========
Governorate
Alexandria    4
Egypt         1
Name: count, dtype: int64

========== City ==========
City
Zizinia       2
Sidi Bishr    1
Nationwide    1
Alexandria    1
Name: count, dtype: int64


In [24]:
# ============================================================
# 2.4 Strip spaces + drop reference-only columns
# ============================================================
for col in get_text_cols(loc):
    loc[col] = loc[col].str.strip()

# Source_URL / Verification_Class -> where the fact came from (audit only)
loc = loc.drop(columns=["Source_URL", "Verification_Class"])

print("Remaining columns:", list(loc.columns))

Remaining columns: ['Location_ID', 'Location_Name', 'Location_Type', 'Channel', 'Governorate', 'City', 'Public_Address']


In [25]:
# ============================================================
# 2.5 Fix Governorate for the online store
# ============================================================
# "Egypt" is a country, not a governorate. The online store delivers nationwide,
# so we label it "Nationwide" (matches its City value) to avoid a fake
mask = loc["Governorate"] == "Egypt"
print("Rows to fix:", mask.sum())

loc.loc[mask, "Governorate"] = "Nationwide"

loc[["Location_ID", "Location_Name", "Channel", "Governorate", "City"]]

Rows to fix: 1


,Location_ID,Location_Name,Channel,Governorate,City
0,LOC001,LEO MODA - Sidi Bishr Store,Retail,Alexandria,Sidi Bishr
1,LOC002,LEO MODA Online Store,Online,Nationwide,Nationwide
2,LOC003,El-Sharnouby Wholesale Desk,Wholesale,Alexandria,Zizinia
3,LOC004,El-Sharnouby Head Office,Corporate,Alexandria,Zizinia
4,LOC005,LEO Production Hub,Operations,Alexandria,Alexandria


In [26]:
# ============================================================
# 2.6 Helper columns for analysis and the dashboard
# ============================================================
# 1) Is_Sales_Point: does this location sell to customers?
#    Retail / Online / Wholesale -> yes
#    Head Office / Factory       -> no (exclude from sales analysis)
SALES_CHANNELS = ["Retail", "Online", "Wholesale"]
loc["Is_Sales_Point"] = loc["Channel"].isin(SALES_CHANNELS)

# 2) Location_Short: short display name for charts and the bot
#    (full names like "El-Sharnouby Wholesale Desk" are too long for chart labels)
short_names = {
    "LOC001": "Sidi Bishr Store",
    "LOC002": "Online Store",
    "LOC003": "Wholesale Desk",
    "LOC004": "Head Office",
    "LOC005": "Production Hub",
}
# .map() looks up each ID in the dictionary;
# .fillna() keeps the full name for any ID not in the dictionary
loc["Location_Short"] = loc["Location_ID"].map(short_names).fillna(loc["Location_Name"])

loc[["Location_ID", "Location_Short", "Channel", "Is_Sales_Point"]]

,Location_ID,Location_Short,Channel,Is_Sales_Point
0,LOC001,Sidi Bishr Store,Retail,True
1,LOC002,Online Store,Online,True
2,LOC003,Wholesale Desk,Wholesale,True
3,LOC004,Head Office,Corporate,False
4,LOC005,Production Hub,Operations,False


In [27]:
loc.head(5)

,Location_ID,Location_Name,Location_Type,Channel,Governorate,City,Public_Address,Is_Sales_Point,Location_Short
0,LOC001,LEO MODA - Sidi Bishr Store,Retail Store,Retail,Alexandria,Sidi Bishr,"72C2+GXJ, Sidi Bishr Bahri, Montaza 2, Alexandria, Egypt",True,Sidi Bishr Store
1,LOC002,LEO MODA Online Store,Digital Store,Online,Nationwide,Nationwide,Online - delivery across Egypt,True,Online Store
2,LOC003,El-Sharnouby Wholesale Desk,B2B / Wholesale,Wholesale,Alexandria,Zizinia,"5 Amin Yehia Street, Zizinia, Alexandria, Egypt",True,Wholesale Desk
3,LOC004,El-Sharnouby Head Office,Head Office,Corporate,Alexandria,Zizinia,"5 Amin Yehia Street, Zizinia, Alexandria, Egypt",False,Head Office
4,LOC005,LEO Production Hub,Factory / Production,Operations,Alexandria,Alexandria,"Alexandria, Egypt",False,Production Hub


In [28]:
# ============================================================
# 2.7 Final review + save
# ============================================================
print("Shape: before", raw["locations"].shape, "-> after", loc.shape)
print("\nColumns & types:")
for col in loc.columns:
    print(f"  {col:<18} {str(loc[col].dtype)}")

print("\nSales points:", loc.loc[loc["Is_Sales_Point"], "Location_Short"].tolist())

loc.to_parquet(CLEAN_DIR / "locations.parquet", index=False)
loc.to_csv(CLEAN_DIR / "locations.csv", index=False, encoding="utf-8-sig")
print("\nSaved: locations.parquet + locations.csv")

Shape: before (5, 9) -> after (5, 9)

Columns & types:
  Location_ID        str
  Location_Name      str
  Location_Type      str
  Channel            str
  Governorate        str
  City               str
  Public_Address     str
  Is_Sales_Point     bool
  Location_Short     str

Sales points: ['Sidi Bishr Store', 'Online Store', 'Wholesale Desk']

Saved: locations.parquet + locations.csv


###  Locations — Cleaning Summary
| Decision | Reason |
|---|---|
| Stripped spaces | Exact matching |
| Dropped `Source_URL`, `Verification_Class` | Audit only |
| Kept `Public_Address` | Bot can answer "where is the wholesale desk?" |
| `Governorate`: "Egypt" → "Nationwide" (online store) | "Egypt" is not a governorate; matches `City` |
| Added `Is_Sales_Point` | Excludes Head Office & Factory from sales analysis |
| Added `Location_Short` | Readable labels for charts and the bot |

**Result:** 5 locations — 3 sales points (Sidi Bishr store, Online, Wholesale desk), Head Office, Production Hub — all in Alexandria.

Customers

In [29]:
# ============================================================
# 3.1 Working copy + first look
# ============================================================
cust = raw["customers"].copy()

print(f"Shape: {cust.shape[0]:,} rows x {cust.shape[1]} cols")
cust.head(10)

Shape: 30,000 rows x 9 cols


,Customer_ID,Customer_Type,Governorate,Age_Band,Gender,Join_Date,Acquisition_Channel,Consent_Status,Data_Class
0,C00001,Retail,Dakahlia,45-54,Male,2026-02-22,Organic Social,Not opted-in,Modeled/anonymized training customer - not actual person...
1,C00002,Retail,Beheira,35-44,Male,2024-09-30,Direct / WhatsApp,Opted-in,Modeled/anonymized training customer - not actual person...
2,C00003,Retail,Dakahlia,25-34,Female,2025-06-27,Organic Social,Opted-in,Modeled/anonymized training customer - not actual person...
3,C00004,Wholesale,Port Said,Business,Business,2024-09-27,Wholesale Sales,Opted-in,Modeled/anonymized training customer - not actual person...
4,C00005,Retail,Gharbia,55+,Female,2025-02-07,Paid Social,Not opted-in,Modeled/anonymized training customer - not actual person...
5,C00006,Retail,Alexandria,35-44,Female,2025-08-19,Store Walk-in,Opted-in,Modeled/anonymized training customer - not actual person...
6,C00007,Retail,Other,25-34,Female,2026-02-15,Paid Social,Opted-in,Modeled/anonymized training customer - not actual person...
7,C00008,Retail,Alexandria,35-44,Male,2026-01-03,Direct / WhatsApp,Opted-in,Modeled/anonymized training customer - not actual person...
8,C00009,Retail,Alexandria,45-54,Male,2025-06-13,Store Walk-in,Opted-in,Modeled/anonymized training customer - not actual person...
9,C00010,Retail,Alexandria,25-34,Female,2025-04-07,Paid Social,Opted-in,Modeled/anonymized training customer - not actual person...


In [30]:
# ============================================================
# 3.2 Types, missing values, duplicates, ID format
# ============================================================
cust.info()

print("\nMissing values per column:")
print(cust.isna().sum())

print("\nDuplicated Customer_ID:", cust["Customer_ID"].duplicated().sum())
print("Fully duplicated rows: ", cust.duplicated().sum())
valid_id = cust["Customer_ID"].str.match(r"^C\d{5}$")
print("Customer_IDs with unexpected format:", (~valid_id).sum())   # ~ = NOT

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Customer_ID          30000 non-null  str  
 1   Customer_Type        30000 non-null  str  
 2   Governorate          30000 non-null  str  
 3   Age_Band             30000 non-null  str  
 4   Gender               30000 non-null  str  
 5   Join_Date            30000 non-null  str  
 6   Acquisition_Channel  30000 non-null  str  
 7   Consent_Status       30000 non-null  str  
 8   Data_Class           30000 non-null  str  
dtypes: str(9)
memory usage: 5.7 MB

Missing values per column:
Customer_ID            0
Customer_Type          0
Governorate            0
Age_Band               0
Gender                 0
Join_Date              0
Acquisition_Channel    0
Consent_Status         0
Data_Class             0
dtype: int64

Duplicated Customer_ID: 0
Fully duplicated rows:  0
Customer_IDs with un

In [31]:
# ============================================================
# 3.3 Distinct values in categorical columns
# ============================================================
category_cols = ["Customer_Type", "Governorate", "Age_Band", "Gender",
                 "Acquisition_Channel", "Consent_Status", "Data_Class"]

for col in category_cols:
    print(f"\n========== {col} ({cust[col].nunique()} distinct) ==========")
    print(cust[col].value_counts(dropna=False))


========== Customer_Type (2 distinct) ==========
Customer_Type
Retail       28349
Wholesale     1651
Name: count, dtype: int64

========== Governorate (13 distinct) ==========
Governorate
Alexandria        8415
Cairo             5383
Giza              3307
Beheira           2374
Dakahlia          1851
Other             1807
Gharbia           1778
Sharqia           1509
Kafr El Sheikh    1197
Monufia            923
Damietta           599
Port Said          575
Suez               282
Name: count, dtype: int64

========== Age_Band (6 distinct) ==========
Age_Band
25-34       10705
35-44        8940
45-54        4299
18-24        2681
55+          1724
Business     1651
Name: count, dtype: int64

========== Gender (4 distinct) ==========
Gender
Female           18931
Male              8548
Business          1651
Not specified      870
Name: count, dtype: int64

========== Acquisition_Channel (7 distinct) ==========
Acquisition_Channel
Direct / WhatsApp    6316
Organic Social       5639
St

In [32]:
# ============================================================
# 3.4 Strip spaces + drop constant column
# ============================================================
for col in get_text_cols(cust):
    cust[col] = cust[col].str.strip()

# Confirm Data_Class is a single repeated sentence before dropping it
print("Data_Class distinct values:", cust["Data_Class"].nunique())
cust = cust.drop(columns=["Data_Class"], errors="ignore")

print("Remaining columns:", list(cust.columns))

Data_Class distinct values: 1
Remaining columns: ['Customer_ID', 'Customer_Type', 'Governorate', 'Age_Band', 'Gender', 'Join_Date', 'Acquisition_Channel', 'Consent_Status']


In [33]:
# ============================================================
# 0.5 Helper: robust date parser
# ============================================================
# Excel DISPLAYS dates in your computer's format, but the CSV may store
# them differently (e.g. "2026-02-22"). So we never trust Excel's display:
# we try several known formats and keep the one that converts the most values.
DATE_FORMATS = [
    "%Y-%m-%d",            # 2026-02-22
    "%m/%d/%Y",            # 2/22/2026
    "%d/%m/%Y",            # 22/2/2026
    "%Y-%m-%d %H:%M:%S",   # 2026-02-22 00:00:00
]

def parse_dates(series, name=""):
    """Try each format; return the best-converted series and report the result."""
    best_result, best_fmt, best_ok = None, None, -1
    for fmt in DATE_FORMATS:
        converted = pd.to_datetime(series, format=fmt, errors="coerce")
        ok = converted.notna().sum()
        if ok > best_ok:
            best_result, best_fmt, best_ok = converted, fmt, ok

    filled = series.notna().sum()   # how many values were filled originally
    failed = filled - best_ok
    print(f"[{name}] format used: {best_fmt}  ->  converted {best_ok:,} of {filled:,}  (failed: {failed:,})")
    if failed > 0:
        bad = series[series.notna() & best_result.isna()]
        print("   examples that failed:", bad.head().tolist())
    return best_result

In [34]:
# ============================================================
# 3.5 Join_Date: text -> real date
# ============================================================
# First, look at the REAL text stored in the file (not Excel's display)
print("Raw Join_Date examples:", raw["customers"]["Join_Date"].head(5).tolist())

# Convert from the ORIGINAL raw column (cust["Join_Date"] may already be empty
# from the previous wrong attempt)
cust["Join_Date"] = parse_dates(raw["customers"]["Join_Date"], name="Join_Date")

print("\nJoin_Date range:", cust["Join_Date"].min().date(), "->", cust["Join_Date"].max().date())

today = pd.Timestamp.today().normalize()
print("Join dates in the future:", (cust["Join_Date"] > today).sum())

print("\nCustomers joined per year:")
print(cust["Join_Date"].dt.year.value_counts().sort_index())

Raw Join_Date examples: ['2026-02-22', '2024-09-30', '2025-06-27', '2024-09-27', '2025-02-07']
[Join_Date] format used: %Y-%m-%d  ->  converted 30,000 of 30,000  (failed: 0)

Join_Date range: 2023-10-01 -> 2026-09-30
Join dates in the future: 0

Customers joined per year:
Join_Date
2023    2580
2024    9970
2025    9901
2026    7549
Name: count, dtype: int64


In [35]:
# ============================================================
# 3.6 "Business" in Age_Band / Gender
# ============================================================
# Question: is "Business" written for EVERY wholesale row, and ONLY for them?
# With 30k rows we can't check by eye -> let the code answer.

is_wholesale = cust["Customer_Type"] == "Wholesale"
age_business = cust["Age_Band"] == "Business"
gen_business = cust["Gender"] == "Business"

# --- Check 1: do ALL wholesale rows have "Business" in BOTH columns?
wholesale_not_business = is_wholesale & ~(age_business & gen_business)
print("Check 1 — Wholesale rows WITHOUT 'Business' in both columns:", wholesale_not_business.sum())

# --- Check 2: does any RETAIL row have "Business"? (should never happen)
retail_with_business = ~is_wholesale & (age_business | gen_business)
print("Check 2 — Retail rows WITH 'Business':", retail_with_business.sum())

# --- Check 3: full picture as a table
print("\nCheck 3 — Customer_Type x Age_Band is 'Business':")
print(pd.crosstab(cust["Customer_Type"], age_business, margins=True))
print("\nCustomer_Type x Gender is 'Business':")
print(pd.crosstab(cust["Customer_Type"], gen_business, margins=True))

# Show any problem rows (expected: both tables empty)
if wholesale_not_business.any():
    print("\nWholesale rows without 'Business':")
    display(cust.loc[wholesale_not_business].head(10))
if retail_with_business.any():
    print("\nRetail rows with 'Business':")
    display(cust.loc[retail_with_business].head(10))

Check 1 — Wholesale rows WITHOUT 'Business' in both columns: 0
Check 2 — Retail rows WITH 'Business': 0

Check 3 — Customer_Type x Age_Band is 'Business':
Age_Band       False  True    All
Customer_Type                    
Retail         28349     0  28349
Wholesale          0  1651   1651
All            28349  1651  30000

Customer_Type x Gender is 'Business':
Gender         False  True    All
Customer_Type                    
Retail         28349     0  28349
Wholesale          0  1651   1651
All            28349  1651  30000


In [36]:
# ============================================================
# 3.6b Replace "Business" with missing (not applicable)
# ============================================================
# Traders are shops/companies, not people -> age and gender don't apply.
# Keeping "Business" would create a fake category in demographic charts
# and in the churn model. Customer_Type still tells us who is a trader.
#
# We blank the value if EITHER it says "Business" OR the customer is Wholesale.
# This also covers any wholesale row that Check 1 found with a real-looking value.
for col in ["Age_Band", "Gender"]:
    to_blank = (cust[col] == "Business") | is_wholesale
    cust[col] = cust[col].mask(to_blank)   # mask -> where True, value becomes missing
    print(f"{col}: {to_blank.sum():,} values set to missing")

print("\nAge_Band after:")
print(cust["Age_Band"].value_counts(dropna=False))
print("\nGender after:")
print(cust["Gender"].value_counts(dropna=False))

# Final check: no "Business" left anywhere
print("\n'Business' remaining:",
      (cust["Age_Band"] == "Business").sum() + (cust["Gender"] == "Business").sum())

Age_Band: 1,651 values set to missing
Gender: 1,651 values set to missing

Age_Band after:
Age_Band
25-34    10705
35-44     8940
45-54     4299
18-24     2681
55+       1724
NaN       1651
Name: count, dtype: int64

Gender after:
Gender
Female           18931
Male              8548
NaN               1651
Not specified      870
Name: count, dtype: int64

'Business' remaining: 0


In [37]:
# ============================================================
# 3.7 Consent_Status -> boolean
# ============================================================
# The bot may only auto-message customers who opted in.
consent_map = {
    "Opted-in":     True,
    "Not opted-in": False,
}
cust["Is_Opted_In"] = cust["Consent_Status"].map(consent_map)

# Values not found in the dictionary become missing -> must be 0
unmapped = cust["Is_Opted_In"].isna() & cust["Consent_Status"].notna()
print("Unmapped consent values:", unmapped.sum())
if unmapped.any():
    print("Values:", cust.loc[unmapped, "Consent_Status"].unique())

# "boolean" (lowercase b) = True/False type that allows missing values
cust["Is_Opted_In"] = cust["Is_Opted_In"].astype("boolean")
cust = cust.drop(columns=["Consent_Status"], errors="ignore")

print("\nOpted-in by Customer_Type:")
print(pd.crosstab(cust["Customer_Type"], cust["Is_Opted_In"], margins=True))

Unmapped consent values: 0

Opted-in by Customer_Type:
Is_Opted_In    False   True    All
Customer_Type                     
Retail         10237  18112  28349
Wholesale        576   1075   1651
All            10813  19187  30000


In [38]:
# ============================================================
# 3.8 Consistency checks
# ============================================================
# 1) Acquisition channel vs customer type
#    Expected: "Wholesale Sales" channel -> Wholesale customers
print("========== Acquisition_Channel x Customer_Type ==========")
print(pd.crosstab(cust["Acquisition_Channel"], cust["Customer_Type"]))

# 2) Governorates sorted alphabetically -> spelling variants appear next to each other
print("\n========== Governorates (alphabetical) ==========")
print(sorted(cust["Governorate"].dropna().unique()))

========== Acquisition_Channel x Customer_Type ==========
Customer_Type        Retail  Wholesale
Acquisition_Channel                   
Direct / WhatsApp      6316          0
Google Search          3401          0
Organic Social         5639          0
Paid Social            5175          0
Referral               2576          0
Store Walk-in          5242          0
Wholesale Sales           0       1651

========== Governorates (alphabetical) ==========
['Alexandria', 'Beheira', 'Cairo', 'Dakahlia', 'Damietta', 'Gharbia', 'Giza', 'Kafr El Sheikh', 'Monufia', 'Other', 'Port Said', 'Sharqia', 'Suez']


In [39]:
# ============================================================
# 3.10 Reduce memory: repeated text -> "category" type
# ============================================================
# "category" stores each distinct value ONCE and a small code per row
before_mb = cust.memory_usage(deep=True).sum() / 1024**2

cat_cols = ["Customer_Type", "Governorate", "Acquisition_Channel", "Age_Band", "Gender"]
for col in cat_cols:
    cust[col] = cust[col].astype("category")

after_mb = cust.memory_usage(deep=True).sum() / 1024**2
print(f"Memory: {before_mb:.2f} MB -> {after_mb:.2f} MB  "
      f"({(1 - after_mb / before_mb) * 100:.0f}% smaller)")

print("\nMemory per column (MB):")
print((cust.memory_usage(deep=True) / 1024**2).round(3))

Memory: 2.91 MB -> 0.83 MB  (71% smaller)

Memory per column (MB):
Index                  0.000
Customer_ID            0.404
Customer_Type          0.029
Governorate            0.029
Age_Band               0.029
Gender                 0.029
Join_Date              0.229
Acquisition_Channel    0.029
Is_Opted_In            0.057
dtype: float64


In [40]:
# ============================================================
# 3.11 Final review
# ============================================================
print("Shape: before", raw["customers"].shape, "-> after", cust.shape)

print("\nColumns & types:")
for col in cust.columns:
    print(f"  {col:<22} {str(cust[col].dtype)}")

print("\nMissing values (Age_Band / Gender missing = wholesale traders, expected):")
print(cust.isna().sum()[cust.isna().sum() > 0])

# Key numbers for the project
is_ws = cust["Customer_Type"] == "Wholesale"
print(f"\nWholesale traders: {is_ws.sum():,}")
print(f"Retail customers:  {(~is_ws).sum():,}")
print(f"Opted-in traders:  {cust.loc[is_ws, 'Is_Opted_In'].sum():,}")

print("\nTop 10 governorates for traders:")
print(cust.loc[is_ws, "Governorate"].value_counts().head(10))

Shape: before (30000, 9) -> after (30000, 8)

Columns & types:
  Customer_ID            str
  Customer_Type          category
  Governorate            category
  Age_Band               category
  Gender                 category
  Join_Date              datetime64[us]
  Acquisition_Channel    category
  Is_Opted_In            boolean

Missing values (Age_Band / Gender missing = wholesale traders, expected):
Age_Band    1651
Gender      1651
dtype: int64

Wholesale traders: 1,651
Retail customers:  28,349
Opted-in traders:  1,075

Top 10 governorates for traders:
Governorate
Alexandria        484
Cairo             295
Giza              194
Beheira           132
Dakahlia           96
Gharbia            94
Other              89
Sharqia            67
Kafr El Sheikh     64
Monufia            43
Name: count, dtype: int64


In [41]:
# ============================================================
# 3.12 Save
# ============================================================
# Parquet keeps datetime, boolean, and category types
cust.to_parquet(CLEAN_DIR / "customers.parquet", index=False)

# CSV for Excel review (dates written as YYYY-MM-DD)
cust.to_csv(CLEAN_DIR / "customers.csv", index=False, encoding="utf-8-sig")

print("Saved: customers.parquet + customers.csv")

Saved: customers.parquet + customers.csv


In [42]:
cust.head(10)

,Customer_ID,Customer_Type,Governorate,Age_Band,Gender,Join_Date,Acquisition_Channel,Is_Opted_In
0,C00001,Retail,Dakahlia,45-54,Male,2026-02-22,Organic Social,False
1,C00002,Retail,Beheira,35-44,Male,2024-09-30,Direct / WhatsApp,True
2,C00003,Retail,Dakahlia,25-34,Female,2025-06-27,Organic Social,True
3,C00004,Wholesale,Port Said,NaN,NaN,2024-09-27,Wholesale Sales,True
4,C00005,Retail,Gharbia,55+,Female,2025-02-07,Paid Social,False
5,C00006,Retail,Alexandria,35-44,Female,2025-08-19,Store Walk-in,True
6,C00007,Retail,Other,25-34,Female,2026-02-15,Paid Social,True
7,C00008,Retail,Alexandria,35-44,Male,2026-01-03,Direct / WhatsApp,True
8,C00009,Retail,Alexandria,45-54,Male,2025-06-13,Store Walk-in,True
9,C00010,Retail,Alexandria,25-34,Female,2025-04-07,Paid Social,True


###  Customers — Cleaning Summary
| Decision | Reason |
|---|---|
| Stripped spaces; validated `Customer_ID` format (`C` + 5 digits) | IDs must match Sales exactly |
| Dropped `Data_Class` | Constant value |
| `Join_Date` text → datetime (`%m/%d/%Y`) | Needed for tenure / recency |
| Verified "Business" appears in Age_Band/Gender for all wholesale rows and no retail rows | Checked by code, not by eye (30k rows) |
| `Age_Band` / `Gender` → missing for wholesale traders | Not applicable to businesses; avoids a fake demographic category |
| `Consent_Status` → `Is_Opted_In` (True/False) | Bot may only auto-message opted-in customers |
| Low-cardinality text → `category` | Memory reduction |

**Notes for later:**
- Tenure (days since `Join_Date`) will be calculated relative to the last date in Sales, not today.
- Customers who joined after the last sale date will have no purchases — check in the Sales notebook.

---
# 4. Inventory_History


In [43]:
# ============================================================
# 4.1 Working copy + first look
# ============================================================
inv = raw["inventory"].copy()

print(f"Shape: {inv.shape[0]:,} rows x {inv.shape[1]} cols")
print("\nRaw Snapshot_Month examples:", inv["Snapshot_Month"].head(3).tolist())
inv.head(10)

Shape: 6,372 rows x 19 cols

Raw Snapshot_Month examples: ['2023-10-01', '2023-11-01', '2023-12-01']


,Snapshot_Month,Location_ID,Location_Name,Product_ID,SKU,Product_Name,Opening_Stock,Received_Qty,Sold_Qty,Returned_Qty,Damaged_Qty,Closing_Stock,Avg_Daily_Sales_90D,Safety_Stock_Units,Reorder_Point_Units,Supplier_Lead_Time_Days,Days_of_Cover,Stockout_Days_In_Month,Data_Class
0,2023-10-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,176,39,42,3,0,176,1.38,5,12,5,127.5,0,Modeled inventory history - no forecast columns provided
1,2023-11-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,176,65,62,4,1,182,1.71,23,59,21,106.4,0,Modeled inventory history - no forecast columns provided
2,2023-12-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,182,54,61,1,1,175,1.81,25,63,21,96.7,0,Modeled inventory history - no forecast columns provided
3,2024-01-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,175,35,42,1,0,169,1.81,12,30,10,93.4,0,Modeled inventory history - no forecast columns provided
4,2024-02-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,169,28,31,1,0,167,1.47,7,17,7,113.6,0,Modeled inventory history - no forecast columns provided
5,2024-03-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,167,105,112,10,0,170,2.03,9,23,7,83.7,0,Modeled inventory history - no forecast columns provided
6,2024-04-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,170,119,127,3,1,164,2.96,40,102,21,55.4,0,Modeled inventory history - no forecast columns provided
7,2024-05-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,164,128,134,3,0,161,4.09,56,142,21,39.4,0,Modeled inventory history - no forecast columns provided
8,2024-06-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,161,150,139,3,0,175,4.39,14,36,5,39.9,0,Modeled inventory history - no forecast columns provided
9,2024-07-01,LOC001,LEO MODA - Sidi Bishr Store,P001,26115-8,Summer children's girls-Happy,175,152,144,6,0,189,4.57,42,106,14,41.4,0,Modeled inventory history - no forecast columns provided


In [44]:
# ============================================================
# 4.2 Types, missing values, composite-key duplicates
# ============================================================
inv.info()

print("\nMissing values per column:")
print(inv.isna().sum())

# Composite key: the combination of these 3 columns must be unique
KEY = ["Snapshot_Month", "Location_ID", "Product_ID"]
dup_key = inv.duplicated(subset=KEY, keep=False)
print(f"\nRows sharing the same (month, location, product): {dup_key.sum():,}")
print("Fully duplicated rows:", inv.duplicated().sum())

# Show examples if any
inv.loc[dup_key].sort_values(KEY).head(10)

<class 'pandas.DataFrame'>
RangeIndex: 6372 entries, 0 to 6371
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Snapshot_Month           6372 non-null   str    
 1   Location_ID              6372 non-null   str    
 2   Location_Name            6372 non-null   str    
 3   Product_ID               6372 non-null   str    
 4   SKU                      6372 non-null   str    
 5   Product_Name             6372 non-null   str    
 6   Opening_Stock            6372 non-null   int64  
 7   Received_Qty             6372 non-null   int64  
 8   Sold_Qty                 6372 non-null   int64  
 9   Returned_Qty             6372 non-null   int64  
 10  Damaged_Qty              6372 non-null   int64  
 11  Closing_Stock            6372 non-null   int64  
 12  Avg_Daily_Sales_90D      6372 non-null   float64
 13  Safety_Stock_Units       6372 non-null   int64  
 14  Reorder_Point_Units      6372 non-n

,Snapshot_Month,Location_ID,Location_Name,Product_ID,SKU,Product_Name,Opening_Stock,Received_Qty,Sold_Qty,Returned_Qty,Damaged_Qty,Closing_Stock,Avg_Daily_Sales_90D,Safety_Stock_Units,Reorder_Point_Units,Supplier_Lead_Time_Days,Days_of_Cover,Stockout_Days_In_Month,Data_Class


In [45]:
# ============================================================
# 4.3 Snapshot_Month: text -> date
# ============================================================
inv["Snapshot_Month"] = parse_dates(raw["inventory"]["Snapshot_Month"], name="Snapshot_Month")

# A monthly snapshot should always be dated on day 1 of the month
not_day_1 = (inv["Snapshot_Month"].dt.day != 1).sum()
print("Snapshots not on day 1:", not_day_1)

first, last = inv["Snapshot_Month"].min(), inv["Snapshot_Month"].max()
expected_months = len(pd.date_range(first, last, freq="MS"))   # MS = Month Start
print(f"\nRange: {first.date()} -> {last.date()}")
print(f"Distinct months in data: {inv['Snapshot_Month'].nunique()}  |  expected in range: {expected_months}")

print("\nRows per month (first and last 3):")
per_month = inv["Snapshot_Month"].value_counts().sort_index()
print(pd.concat([per_month.head(3), per_month.tail(3)]))

[Snapshot_Month] format used: %Y-%m-%d  ->  converted 6,372 of 6,372  (failed: 0)
Snapshots not on day 1: 0

Range: 2023-10-01 -> 2026-09-01
Distinct months in data: 36  |  expected in range: 36

Rows per month (first and last 3):
Snapshot_Month
2023-10-01    177
2023-11-01    177
2023-12-01    177
2026-07-01    177
2026-08-01    177
2026-09-01    177
Name: count, dtype: int64


In [46]:
# ============================================================
# 4.4 Referential integrity: do all IDs exist in the reference files?
# ============================================================
# Every Product_ID / Location_ID here must exist in products / locations,
# otherwise these rows will "disappear" when we join tables later.
inv_products = set(inv["Product_ID"])
inv_locations = set(inv["Location_ID"])

missing_products  = inv_products  - set(prod["Product_ID"])
missing_locations = inv_locations - set(loc["Location_ID"])
print("Product_IDs not in Product_Master: ", sorted(missing_products))
print("Location_IDs not in Locations:     ", sorted(missing_locations))

# The other direction: products in the catalog that never appear in inventory
never_stocked = set(prod["Product_ID"]) - inv_products
print("\nCatalog products with no inventory history:", sorted(never_stocked))

# Which locations hold stock? (use the short names from section 2)
print("\nRows per location:")
loc_names = loc.set_index("Location_ID")["Location_Short"]
print(inv["Location_ID"].map(loc_names).value_counts())

print(f"\nProducts per location:")
print(inv.groupby("Location_ID")["Product_ID"].nunique().rename(index=loc_names))

Product_IDs not in Product_Master:  []
Location_IDs not in Locations:      []

Catalog products with no inventory history: []

Rows per location:
Location_ID
Sidi Bishr Store    2124
Online Store        2124
Wholesale Desk      2124
Name: count, dtype: int64

Products per location:
Location_ID
Sidi Bishr Store    59
Online Store        59
Wholesale Desk      59
Name: Product_ID, dtype: int64


In [47]:
# ============================================================
# 4.5 Drop columns that duplicate the reference files
# ============================================================
# Location_Name -> already in locations (join on Location_ID)
# SKU, Product_Name -> already in products (join on Product_ID)
# Data_Class -> constant sentence
# Keeping descriptive text in ONE place (the reference file) avoids
# inconsistencies, e.g. the same product spelled two different ways.
print("Data_Class distinct values:", inv["Data_Class"].nunique())

inv = inv.drop(columns=["Location_Name", "SKU", "Product_Name", "Data_Class"], errors="ignore")
print("Remaining columns:", list(inv.columns))

Data_Class distinct values: 1
Remaining columns: ['Snapshot_Month', 'Location_ID', 'Product_ID', 'Opening_Stock', 'Received_Qty', 'Sold_Qty', 'Returned_Qty', 'Damaged_Qty', 'Closing_Stock', 'Avg_Daily_Sales_90D', 'Safety_Stock_Units', 'Reorder_Point_Units', 'Supplier_Lead_Time_Days', 'Days_of_Cover', 'Stockout_Days_In_Month']


In [48]:
# ============================================================
# 4.6 Numeric sanity: types and impossible values
# ============================================================
qty_cols = ["Opening_Stock", "Received_Qty", "Sold_Qty", "Returned_Qty", "Damaged_Qty",
            "Closing_Stock", "Safety_Stock_Units", "Reorder_Point_Units"]
rate_cols = ["Avg_Daily_Sales_90D", "Supplier_Lead_Time_Days", "Days_of_Cover",
             "Stockout_Days_In_Month"]

# Force numeric (safety step)
for col in qty_cols + rate_cols:
    inv[col] = pd.to_numeric(inv[col], errors="coerce")

# Negative values are impossible for stock quantities and rates
print("Negative values per column:")
negatives = (inv[qty_cols + rate_cols] < 0).sum()
print(negatives[negatives > 0] if negatives.any() else "None")

# Stockout days can't exceed the number of days in that month
too_many = inv["Stockout_Days_In_Month"] > inv["Snapshot_Month"].dt.days_in_month
print("\nStockout days > days in month:", too_many.sum())

print("\nSummary of key columns:")
print(inv[qty_cols + rate_cols].describe().T.round(1))

Negative values per column:
None

Stockout days > days in month: 0

Summary of key columns:
                          count   mean     std  min    25%    50%    75%      max
Opening_Stock            6372.0  358.7   493.8  0.0  135.0  222.0  341.0   3696.0
Received_Qty             6372.0  257.4   451.6  0.0   25.0   64.0  264.2   4011.0
Sold_Qty                 6372.0  260.6   465.7  0.0   23.0   56.0  264.0   4500.0
Returned_Qty             6372.0    5.8    14.7  0.0    0.0    1.0    4.0    186.0
Damaged_Qty              6372.0    1.0     1.8  0.0    0.0    0.0    1.0     18.0
Closing_Stock            6372.0  362.9   511.0  0.0  133.0  222.0  339.2   3789.0
Safety_Stock_Units       6372.0   63.2   123.0  5.0    5.0   13.0   61.0   1467.0
Reorder_Point_Units      6372.0  159.7   312.6  5.0   13.0   33.0  154.2   3724.0
Avg_Daily_Sales_90D      6372.0    8.5    14.6  0.0    0.8    1.8    9.9    133.2
Supplier_Lead_Time_Days  6372.0   11.4     5.7  5.0    7.0   10.0   14.0     21.0
Days_o

In [49]:
# ============================================================
# 4.7 Stock equation check (row level)
# Opening + Received - Sold + Returned - Damaged = Closing
# ============================================================
calculated_close = (inv["Opening_Stock"] + inv["Received_Qty"] - inv["Sold_Qty"]
                    + inv["Returned_Qty"] - inv["Damaged_Qty"])
equation_gap = calculated_close - inv["Closing_Stock"]

bad_equation = equation_gap != 0
print(f"Rows where the stock equation does NOT hold: {bad_equation.sum():,} of {len(inv):,}")

if bad_equation.any():
    print("\nGap distribution (calculated - recorded):")
    print(equation_gap[bad_equation].describe().round(1))
    display(inv.loc[bad_equation].assign(Gap=equation_gap[bad_equation]).head(10))

Rows where the stock equation does NOT hold: 147 of 6,372

Gap distribution (calculated - recorded):
count     147.0
mean     -106.8
std       242.3
min     -2108.0
25%       -97.5
50%       -47.0
75%       -18.0
max        -1.0
dtype: float64


,Snapshot_Month,Location_ID,Product_ID,Opening_Stock,Received_Qty,Sold_Qty,Returned_Qty,Damaged_Qty,Closing_Stock,Avg_Daily_Sales_90D,Safety_Stock_Units,Reorder_Point_Units,Supplier_Lead_Time_Days,Days_of_Cover,Stockout_Days_In_Month,Gap
32,2026-06-01,LOC001,P001,49,118,202,4,1,0,6.70,30,77,7,0.0,6,-32
33,2026-07-01,LOC001,P001,0,117,196,4,0,0,6.44,59,149,14,0.0,5,-75
69,2026-07-01,LOC001,P002,11,52,93,6,0,0,2.73,37,94,21,0.0,3,-24
2193,2026-07-01,LOC002,P002,37,70,127,9,1,0,3.96,36,91,14,0.0,9,-12
2194,2026-08-01,LOC002,P002,0,94,98,2,0,0,3.71,17,43,7,0.0,12,-2
2195,2026-09-01,LOC002,P002,0,55,62,6,0,0,3.15,43,109,21,0.0,9,-1
4258,2024-08-01,LOC003,P001,12,2186,2310,78,10,0,83.42,380,964,7,0.0,8,-44
4259,2024-09-01,LOC003,P001,0,1510,1656,66,6,0,74.80,681,1728,14,0.0,9,-86
4260,2024-10-01,LOC003,P001,0,647,672,0,1,0,50.86,165,419,5,0.0,3,-26
4262,2024-12-01,LOC003,P001,22,1055,1218,144,7,0,28.88,263,667,14,0.0,12,-4


In [53]:
# ============================================================
# 4.7b Investigate the 147 rows: is it a stockout effect?
# ============================================================
# Hypothesis: customers wanted MORE than was available, so Closing was
# floored at 0, but Sold_Qty kept the full demand (which is impossible to sell).
available = (inv["Opening_Stock"] + inv["Received_Qty"]
             + inv["Returned_Qty"] - inv["Damaged_Qty"])   # units that could be sold
calc_close = available - inv["Sold_Qty"]
bad = calc_close != inv["Closing_Stock"]

print(f"Rows that break the equation:            {bad.sum()}")
print(f"  ...with Closing_Stock = 0:              {(bad & (inv['Closing_Stock'] == 0)).sum()}")
print(f"  ...where Sold_Qty > available stock:    {(bad & (inv['Sold_Qty'] > available)).sum()}")
print(f"  ...with Stockout_Days > 0:              {(bad & (inv['Stockout_Days_In_Month'] > 0)).sum()}")

all_three = bad & (inv["Closing_Stock"] == 0) & (inv["Sold_Qty"] > available) \
                & (inv["Stockout_Days_In_Month"] > 0)
print(f"\nRows matching ALL three signs: {all_three.sum()}  (if = {bad.sum()}, hypothesis confirmed)")

# Reverse check: stockout months where the equation DOES hold
print(f"Stockout months that respect the equation: {((inv['Stockout_Days_In_Month'] > 0) & ~bad).sum()}")

Rows that break the equation:            147
  ...with Closing_Stock = 0:              147
  ...where Sold_Qty > available stock:    147
  ...with Stockout_Days > 0:              147

Rows matching ALL three signs: 147  (if = 147, hypothesis confirmed)
Stockout months that respect the equation: 0


In [54]:
# ============================================================
# 4.7c Fix: split Sold_Qty into what was actually sold vs. lost demand
# ============================================================
# Demand_Qty     = what customers wanted (the original Sold_Qty)
# Sold_Qty       = what could actually be sold (capped at available stock)
# Lost_Sales_Qty = demand that couldn't be served because of the stockout
# Forecasting should use Demand_Qty — otherwise stockouts make demand look
# smaller than it really was, and we'd repeat the same shortage.
inv["Demand_Qty"] = inv["Sold_Qty"]

over_sold = inv["Sold_Qty"] > available
inv.loc[over_sold, "Sold_Qty"] = available[over_sold]

inv["Lost_Sales_Qty"] = inv["Demand_Qty"] - inv["Sold_Qty"]

# Re-check the stock equation -> must be 0 now
recheck = (inv["Opening_Stock"] + inv["Received_Qty"] - inv["Sold_Qty"]
           + inv["Returned_Qty"] - inv["Damaged_Qty"]) != inv["Closing_Stock"]
print(f"Rows fixed:                     {over_sold.sum()}")
print(f"Equation breaks after the fix:  {recheck.sum()}")
print(f"\nTotal lost sales: {inv['Lost_Sales_Qty'].sum():,.0f} units "
      f"({inv['Lost_Sales_Qty'].sum() / inv['Demand_Qty'].sum() * 100:.1f}% of total demand)")

# Which products lost the most sales?
print("\nTop 5 products by lost sales:")
print(inv.groupby("Product_ID")["Lost_Sales_Qty"].sum().sort_values(ascending=False).head(5))

Rows fixed:                     147
Equation breaks after the fix:  0

Total lost sales: 15,704 units (0.9% of total demand)

Top 5 products by lost sales:
Product_ID
P001    5068
P002    1772
P045    1286
P049     938
P017     919
Name: Lost_Sales_Qty, dtype: int64


In [50]:
# ============================================================
# 4.8 Continuity: this month's Opening = last month's Closing
# ============================================================
inv = inv.sort_values(["Location_ID", "Product_ID", "Snapshot_Month"]).reset_index(drop=True)

group = inv.groupby(["Location_ID", "Product_ID"])
prev_close = group["Closing_Stock"].shift(1)    # last month's closing, per product-location
prev_month = group["Snapshot_Month"].shift(1)   # last month's date

# 1) Opening should equal the previous month's closing
has_prev = prev_close.notna()
continuity_gap = inv["Opening_Stock"] - prev_close
bad_continuity = has_prev & (continuity_gap != 0)
print(f"Rows where Opening != previous Closing: {bad_continuity.sum():,}")

# 2) Months should be consecutive (no missing month in the middle)
month_index      = inv["Snapshot_Month"].dt.year * 12 + inv["Snapshot_Month"].dt.month
prev_month_index = prev_month.dt.year * 12 + prev_month.dt.month
month_jump = month_index - prev_month_index
gaps = has_prev & (month_jump != 1)
print(f"Rows with a missing month before them: {gaps.sum():,}")

# How many months does each product-location have?
print("\nMonths per product-location:")
print(group.size().describe().round(1))

Rows where Opening != previous Closing: 0
Rows with a missing month before them: 0

Months per product-location:
count    177.0
mean      36.0
std        0.0
min       36.0
25%       36.0
50%       36.0
75%       36.0
max       36.0
dtype: float64


In [51]:
# ============================================================
# 4.9 Planning formulas check (small rounding differences are OK)
#   Days_of_Cover        = Closing_Stock / Avg_Daily_Sales_90D
#   Reorder_Point_Units  = Avg_Daily_Sales_90D * Supplier_Lead_Time_Days + Safety_Stock_Units
# ============================================================
sells = inv["Avg_Daily_Sales_90D"] > 0
print(f"Rows with zero average daily sales: {(~sells).sum():,}  (Days_of_Cover can't be computed)")

doc_calc = inv["Closing_Stock"] / inv["Avg_Daily_Sales_90D"]
doc_off = sells & ((doc_calc - inv["Days_of_Cover"]).abs() > 1)
print(f"Days_of_Cover off by more than 1 day: {doc_off.sum():,}")

rop_calc = inv["Avg_Daily_Sales_90D"] * inv["Supplier_Lead_Time_Days"] + inv["Safety_Stock_Units"]
rop_off = (rop_calc - inv["Reorder_Point_Units"]).abs() > 1.5
print(f"Reorder_Point off by more than 1.5 units: {rop_off.sum():,}")

if doc_off.any():
    display(inv.loc[doc_off, ["Snapshot_Month", "Product_ID", "Closing_Stock",
                              "Avg_Daily_Sales_90D", "Days_of_Cover"]]
              .assign(Calculated=doc_calc[doc_off].round(1)).head(5))

Rows with zero average daily sales: 0  (Days_of_Cover can't be computed)
Days_of_Cover off by more than 1 day: 0
Reorder_Point off by more than 1.5 units: 0


In [ ]:
# ============================================================
# 4.10 Reduce memory + tidy column order
# ============================================================
before_mb = inv.memory_usage(deep=True).sum() / 1024**2

# IDs repeat many times -> category
for col in ["Location_ID", "Product_ID"]:
    inv[col] = inv[col].astype("category")

column_order = ["Snapshot_Month", "Location_ID", "Product_ID",
                "Opening_Stock", "Received_Qty", "Sold_Qty", "Returned_Qty",
                "Damaged_Qty", "Closing_Stock",
                "Demand_Qty", "Lost_Sales_Qty",
                "Avg_Daily_Sales_90D", "Days_of_Cover", "Stockout_Days_In_Month",
                "Safety_Stock_Units", "Reorder_Point_Units", "Supplier_Lead_Time_Days"]
inv = inv[column_order]

after_mb = inv.memory_usage(deep=True).sum() / 1024**2
print(f"Memory: {before_mb:.2f} MB -> {after_mb:.2f} MB")

Memory: 0.79 MB -> 0.64 MB


In [69]:
# ============================================================
# 4.11 Final review + save
# ============================================================
print("Shape: before", raw["inventory"].shape, "-> after", inv.shape)

print("\nColumns & types:")
for col in inv.columns:
    print(f"  {col:<26} {str(inv[col].dtype)}")

remaining_nulls = inv.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nRemaining missing values:", dict(remaining_nulls) if len(remaining_nulls) else "None")

print(f"\nPeriod: {inv['Snapshot_Month'].min().date()} -> {inv['Snapshot_Month'].max().date()}")
print(f"Products: {inv['Product_ID'].nunique()}  |  Locations: {inv['Location_ID'].nunique()}")

inv.to_parquet(CLEAN_DIR / "inventory.parquet", index=False)
inv.to_csv(CLEAN_DIR / "inventory.csv", index=False, encoding="utf-8-sig")
print("\nSaved: inventory.parquet + inventory.csv")

Shape: before (6372, 19) -> after (6372, 17)

Columns & types:
  Snapshot_Month             datetime64[us]
  Location_ID                category
  Product_ID                 category
  Opening_Stock              int64
  Received_Qty               int64
  Sold_Qty                   int64
  Returned_Qty               int64
  Damaged_Qty                int64
  Closing_Stock              int64
  Avg_Daily_Sales_90D        float64
  Days_of_Cover              float64
  Stockout_Days_In_Month     int64
  Safety_Stock_Units         int64
  Reorder_Point_Units        int64
  Supplier_Lead_Time_Days    int64
  Demand_Qty                 int64
  Lost_Sales_Qty             int64

Remaining missing values: None

Period: 2023-10-01 -> 2026-09-01
Products: 59  |  Locations: 3

Saved: inventory.parquet + inventory.csv


###  Inventory_History — Cleaning Summary
| Check / Decision | Result / Reason |
|---|---|
| Composite key (month + location + product) unique | 0 duplicates |
| `Snapshot_Month` → datetime | 36 consecutive months, all on day 1, no gaps |
| Referential integrity | All IDs exist; stock held at 3 locations (Sidi Bishr, Online, Wholesale Desk) |
| Dropped `Location_Name`, `SKU`, `Product_Name`, `Data_Class` | Normalization: text lives in reference files |
| No negative values; stockout days ≤ days in month | Passed |
| Stock equation | 147 rows (2.3%) broke it: Sold_Qty > available stock in stockout months, Closing floored at 0 |
| Fix | `Demand_Qty` = original Sold_Qty; `Sold_Qty` capped at available; `Lost_Sales_Qty` = difference. Equation now holds on all rows |
| Continuity (Opening = previous Closing) | 0 breaks |
| Days_of_Cover and Reorder_Point formulas | Match on all rows |

**Key insights:**
- Median Days_of_Cover = 96 days → widespread overstock (healthy apparel range: 30–60 days)
- Stockouts also exist → overstock and lost sales at the same time (the size-planning problem)
- Forecasting will use `Demand_Qty`, not `Sold_Qty`

---
# 5. Production_Orders


In [55]:
# ============================================================
# 5.1 Working copy + first look
# ============================================================
prodn = raw["production"].copy()

print(f"Shape: {prodn.shape[0]:,} rows x {prodn.shape[1]} cols")
print("\nAll columns:", list(prodn.columns))
print("\nRaw date examples:")
for col in ["Start_Date", "Due_Date", "Completion_Date"]:
    print(f"  {col}: {prodn[col].head(3).tolist()}")
prodn.head(10)

Shape: 2,500 rows x 16 cols

All columns: ['Production_Order_ID', 'Product_ID', 'Product_Name', 'Start_Date', 'Due_Date', 'Completion_Date', 'Planned_Qty', 'Produced_Qty', 'Good_Qty', 'Rejected_Qty', 'Line_ID', 'Efficiency_Pct', 'Delay_Days', 'Primary_Bottleneck', 'Overtime_Hours', 'Data_Class']

Raw date examples:
  Start_Date: ['2025-11-11', '2025-09-01', '2025-03-22']
  Due_Date: ['2025-11-22', '2025-09-19', '2025-04-08']
  Completion_Date: ['2025-11-22', '2025-09-19', '2025-04-15']


,Production_Order_ID,Product_ID,Product_Name,Start_Date,Due_Date,Completion_Date,Planned_Qty,Produced_Qty,Good_Qty,Rejected_Qty,Line_ID,Efficiency_Pct,Delay_Days,Primary_Bottleneck,Overtime_Hours,Data_Class
0,PO00001,P035,Girls' sweatshirt-Fuchsia,2025-11-11,2025-11-22,2025-11-22,1228,1140,1124,16,LINE-05,81.1,0,NaN,4.5,Modeled factory production order
1,PO00002,P055,Scooby-Doo - بيجامة حريمي صيفي2024,2025-09-01,2025-09-19,2025-09-19,1215,1215,1185,30,LINE-04,89.7,0,NaN,0.5,Modeled factory production order
2,PO00003,P001,Summer children's girls-Happy,2025-03-22,2025-04-08,2025-04-15,613,580,562,18,LINE-04,86.5,7,Finishing,14.7,Modeled factory production order
3,PO00004,P001,Summer children's girls-Happy,2024-05-06,2024-05-22,2024-05-22,4533,3640,3557,83,LINE-03,72.2,0,NaN,3.6,Modeled factory production order
4,PO00005,P045,Girls' autumn pajamas-Stitch,2025-08-08,2025-08-22,2025-08-22,987,756,746,10,LINE-02,72.5,0,NaN,13.6,Modeled factory production order
5,PO00006,P055,Scooby-Doo - بيجامة حريمي صيفي2024,2026-05-04,2026-05-16,2026-05-21,1179,1149,1113,36,LINE-03,83.9,5,Packing,21.4,Modeled factory production order
6,PO00007,P002,Summer children's girls-turtle,2026-09-07,2026-09-18,2026-09-23,2015,1571,1548,23,LINE-01,67.9,5,Packing,14.6,Modeled factory production order
7,PO00008,P050,Yellow - طقم تيشيرت وشورت اطفال صيفي,2024-03-14,2024-04-06,2024-04-06,939,939,921,18,LINE-02,89.8,0,NaN,0.0,Modeled factory production order
8,PO00009,P004,Summer children's girls-Hello kitty black&white,2025-06-28,2025-07-11,2025-07-11,2130,1849,1813,36,LINE-03,80.2,0,NaN,3.8,Modeled factory production order
9,PO00010,P048,Semon - طقم تيشيرت وشورت اطفال صيفي,2025-04-24,2025-05-11,2025-05-11,1195,1151,1125,26,LINE-02,84.3,0,NaN,0.0,Modeled factory production order


In [56]:
# ============================================================
# 5.2 Types, missing values, duplicates
# ============================================================
prodn.info()

print("\nMissing values per column:")
print(prodn.isna().sum())

print("\nDuplicated Production_Order_ID:", prodn["Production_Order_ID"].duplicated().sum())
print("Fully duplicated rows:          ", prodn.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Production_Order_ID  2500 non-null   str    
 1   Product_ID           2500 non-null   str    
 2   Product_Name         2500 non-null   str    
 3   Start_Date           2500 non-null   str    
 4   Due_Date             2500 non-null   str    
 5   Completion_Date      2500 non-null   str    
 6   Planned_Qty          2500 non-null   int64  
 7   Produced_Qty         2500 non-null   int64  
 8   Good_Qty             2500 non-null   int64  
 9   Rejected_Qty         2500 non-null   int64  
 10  Line_ID              2500 non-null   str    
 11  Efficiency_Pct       2500 non-null   float64
 12  Delay_Days           2500 non-null   int64  
 13  Primary_Bottleneck   407 non-null    str    
 14  Overtime_Hours       2500 non-null   float64
 15  Data_Class           2500 non-null   str    
dtyp

In [57]:
# ============================================================
# 5.3 Distinct values in categorical columns
# ============================================================
# Raw text of Primary_Bottleneck BEFORE pandas interpretation:
# re-read just this column with keep_default_na=False so "None" stays as text
raw_bottleneck = pd.read_csv(RAW_DIR / "Production_Orders.csv",
                             usecols=["Primary_Bottleneck"],
                             keep_default_na=False, encoding="utf-8-sig")["Primary_Bottleneck"]
print("========== Primary_Bottleneck (raw text, as written in the file) ==========")
print(raw_bottleneck.value_counts())

print("\n========== Primary_Bottleneck (as pandas loaded it) ==========")
print(prodn["Primary_Bottleneck"].value_counts(dropna=False))

for col in ["Line_ID"] + [c for c in ["Data_Class"] if c in prodn.columns]:
    print(f"\n========== {col} ({prodn[col].nunique()} distinct) ==========")
    print(prodn[col].value_counts(dropna=False))

========== Primary_Bottleneck (raw text, as written in the file) ==========
Primary_Bottleneck
None         2093
Sewing        134
Finishing     109
Packing        61
Cutting        59
Printing       44
Name: count, dtype: int64

========== Primary_Bottleneck (as pandas loaded it) ==========
Primary_Bottleneck
NaN          2093
Sewing        134
Finishing     109
Packing        61
Cutting        59
Printing       44
Name: count, dtype: int64

========== Line_ID (5 distinct) ==========
Line_ID
LINE-01    525
LINE-04    502
LINE-02    501
LINE-03    489
LINE-05    483
Name: count, dtype: int64

========== Data_Class (1 distinct) ==========
Data_Class
Modeled factory production order    2500
Name: count, dtype: int64


In [58]:
# ============================================================
# 5.4 Strip spaces, fix the "None" trap, drop duplicated text columns
# ============================================================
for col in get_text_cols(prodn):
    prodn[col] = prodn[col].str.strip()

# pandas reads the word "None" as missing by default.
# In this column "None" means "no bottleneck" — a real answer, not missing data.
# We use a clearer label so nobody confuses it with missing.
n_fixed = prodn["Primary_Bottleneck"].isna().sum()
prodn["Primary_Bottleneck"] = prodn["Primary_Bottleneck"].fillna("No bottleneck")
print(f"Primary_Bottleneck: {n_fixed:,} empty values -> 'No bottleneck'")
print(prodn["Primary_Bottleneck"].value_counts())

# Product_Name lives in products (normalization); Data_Class is constant
prodn = prodn.drop(columns=["Product_Name", "Data_Class"], errors="ignore")
print("\nRemaining columns:", list(prodn.columns))

Primary_Bottleneck: 2,093 empty values -> 'No bottleneck'
Primary_Bottleneck
No bottleneck    2093
Sewing            134
Finishing         109
Packing            61
Cutting            59
Printing           44
Name: count, dtype: int64

Remaining columns: ['Production_Order_ID', 'Product_ID', 'Start_Date', 'Due_Date', 'Completion_Date', 'Planned_Qty', 'Produced_Qty', 'Good_Qty', 'Rejected_Qty', 'Line_ID', 'Efficiency_Pct', 'Delay_Days', 'Primary_Bottleneck', 'Overtime_Hours']


In [59]:
# ============================================================
# 5.5 Dates: text -> datetime
# ============================================================
date_cols = ["Start_Date", "Due_Date", "Completion_Date"]
for col in date_cols:
    prodn[col] = parse_dates(raw["production"][col], name=col)

print()
for col in date_cols:
    print(f"{col:<16} {prodn[col].min().date()} -> {prodn[col].max().date()}   "
          f"missing: {prodn[col].isna().sum()}")

[Start_Date] format used: %Y-%m-%d  ->  converted 2,500 of 2,500  (failed: 0)
[Due_Date] format used: %Y-%m-%d  ->  converted 2,500 of 2,500  (failed: 0)
[Completion_Date] format used: %Y-%m-%d  ->  converted 2,500 of 2,500  (failed: 0)

Start_Date       2023-10-01 -> 2026-09-10   missing: 0
Due_Date         2023-10-09 -> 2026-10-01   missing: 0
Completion_Date  2023-10-09 -> 2026-09-30   missing: 0


In [60]:
# ============================================================
# 5.6 Date logic checks
# ============================================================
is_open = prodn["Completion_Date"].isna()
print(f"Open orders (not completed yet): {is_open.sum():,}")

# 1) The planned end can't be before the start
bad_due = prodn["Due_Date"] < prodn["Start_Date"]
print(f"Due_Date before Start_Date:          {bad_due.sum()}")

# 2) Completion can't be before the start
bad_done = prodn["Completion_Date"] < prodn["Start_Date"]
print(f"Completion_Date before Start_Date:   {bad_done.sum()}")

# 3) Delay_Days = days late vs. Due_Date (0 if finished on time or early)
delay_calc = (prodn["Completion_Date"] - prodn["Due_Date"]).dt.days.clip(lower=0)
delay_off = ~is_open & (delay_calc != prodn["Delay_Days"])
print(f"Delay_Days doesn't match dates:      {delay_off.sum()}")

if delay_off.any():
    display(prodn.loc[delay_off, ["Production_Order_ID", "Due_Date", "Completion_Date", "Delay_Days"]]
                 .assign(Calculated=delay_calc[delay_off]).head(10))

# 4) Delays vs. bottlenecks: orders with a bottleneck should be the delayed ones
print("\nAverage delay by bottleneck:")
print(prodn.groupby("Primary_Bottleneck")["Delay_Days"].agg(["count", "mean"]).round(1))

Open orders (not completed yet): 0
Due_Date before Start_Date:          0
Completion_Date before Start_Date:   0
Delay_Days doesn't match dates:      0

Average delay by bottleneck:
                    count  mean
Primary_Bottleneck             
Cutting                59   4.2
Finishing             109   5.3
No bottleneck        2093   0.0
Packing                61   4.3
Printing               44   4.4
Sewing                134   7.1


In [61]:
# ============================================================
# 5.7 Quantity logic checks
# ============================================================
qty_cols = ["Planned_Qty", "Produced_Qty", "Good_Qty", "Rejected_Qty",
            "Efficiency_Pct", "Delay_Days", "Overtime_Hours"]
for col in qty_cols:
    prodn[col] = pd.to_numeric(prodn[col], errors="coerce")

# 1) No negatives
negatives = (prodn[qty_cols] < 0).sum()
print("Negative values:", dict(negatives[negatives > 0]) if negatives.any() else "None")

# 2) Good + Rejected must equal Produced
qty_gap = prodn["Good_Qty"] + prodn["Rejected_Qty"] - prodn["Produced_Qty"]
print(f"Good + Rejected != Produced:   {(qty_gap != 0).sum()}")

# 3) Produced vs Planned (producing MORE than planned is unusual -> just report)
print(f"Produced > Planned:            {(prodn['Produced_Qty'] > prodn['Planned_Qty']).sum()}")

# 4) Efficiency must be a percentage between 0 and 100
print(f"Efficiency outside 0–100:      {((prodn['Efficiency_Pct'] < 0) | (prodn['Efficiency_Pct'] > 100)).sum()}")

# 5) Open orders: what has been produced so far?
print("\nOpen orders — quantities so far:")
print(prodn.loc[is_open, ["Planned_Qty", "Produced_Qty", "Good_Qty"]].describe().round(0))

print("\nSummary:")
print(prodn[qty_cols].describe().T.round(1))

Negative values: None
Good + Rejected != Produced:   0
Produced > Planned:            0
Efficiency outside 0–100:      0

Open orders — quantities so far:
       Planned_Qty  Produced_Qty  Good_Qty
count          0.0           0.0       0.0
mean           NaN           NaN       NaN
std            NaN           NaN       NaN
min            NaN           NaN       NaN
25%            NaN           NaN       NaN
50%            NaN           NaN       NaN
75%            NaN           NaN       NaN
max            NaN           NaN       NaN

Summary:
                 count    mean     std    min     25%     50%     75%     max
Planned_Qty     2500.0  2759.0  1418.6  300.0  1512.5  2749.0  4028.2  5200.0
Produced_Qty    2500.0  2507.1  1313.2  228.0  1365.2  2484.5  3586.0  5186.0
Good_Qty        2500.0  2444.6  1281.0  224.0  1331.0  2423.5  3478.0  5129.0
Rejected_Qty    2500.0    62.5    46.3    1.0    25.0    52.0    90.0   265.0
Efficiency_Pct  2500.0    82.2     8.6   48.3    76.5    8

In [62]:
# ============================================================
# 5.8 Referential integrity: products exist in the catalog
# ============================================================
missing_products = set(prodn["Product_ID"]) - set(prod["Product_ID"])
print("Product_IDs not in Product_Master:", sorted(missing_products))

never_produced = set(prod["Product_ID"]) - set(prodn["Product_ID"])
print("Catalog products never produced:   ", sorted(never_produced))

print(f"\nOrders per product: min {prodn.groupby('Product_ID').size().min()}, "
      f"max {prodn.groupby('Product_ID').size().max()}")

Product_IDs not in Product_Master: []
Catalog products never produced:    []

Orders per product: min 7, max 117


In [63]:
# ============================================================
# 5.9 Helper columns (simple facts, no business thresholds)
# ============================================================
# Is_Open        -> order not completed yet (the bot uses this: "next batch coming")
# Lead_Time_Days -> how long an order actually took (start -> completion)
# Planned_Days   -> how long it was supposed to take (start -> due)
# Fill_Rate      -> share of the plan that was actually produced
# Reject_Rate    -> share of produced pieces that were rejected
prodn["Is_Open"]        = prodn["Completion_Date"].isna()
prodn["Lead_Time_Days"] = (prodn["Completion_Date"] - prodn["Start_Date"]).dt.days
prodn["Planned_Days"]   = (prodn["Due_Date"] - prodn["Start_Date"]).dt.days
prodn["Fill_Rate"]      = (prodn["Produced_Qty"] / prodn["Planned_Qty"]).round(3)
prodn["Reject_Rate"]    = (prodn["Rejected_Qty"] / prodn["Produced_Qty"]).round(4)

print("Planned vs actual duration (completed orders):")
print(prodn.loc[~prodn["Is_Open"], ["Planned_Days", "Lead_Time_Days"]].describe().round(1))

print(f"\nAverage fill rate:   {prodn['Fill_Rate'].mean():.1%}")
print(f"Average reject rate: {prodn['Reject_Rate'].mean():.2%}")

Planned vs actual duration (completed orders):
       Planned_Days  Lead_Time_Days
count        2500.0          2500.0
mean           15.4            16.3
std             5.2             5.6
min             7.0             7.0
25%            11.0            12.0
50%            15.0            16.0
75%            20.0            21.0
max            24.0            36.0

Average fill rate:   91.0%
Average reject rate: 2.50%


In [64]:
# ============================================================
# 5.10 Factory capacity check (from Business_Units: up to 5,000 pieces/day)
# ============================================================
# Approximate monthly capacity: 5,000 pieces/day x 26 working days
DAILY_CAPACITY = 5_000
WORKING_DAYS   = 26
MONTHLY_CAPACITY = DAILY_CAPACITY * WORKING_DAYS

done = prodn.loc[~prodn["Is_Open"]].copy()
monthly_output = done.groupby(done["Completion_Date"].dt.to_period("M"))["Produced_Qty"].sum()

print(f"Monthly capacity (approx.): {MONTHLY_CAPACITY:,} pieces")
print(f"Average monthly output:     {monthly_output.mean():,.0f} pieces "
      f"({monthly_output.mean() / MONTHLY_CAPACITY:.0%} of capacity)")
print(f"Busiest month:              {monthly_output.idxmax()}  ->  {monthly_output.max():,.0f} pieces "
      f"({monthly_output.max() / MONTHLY_CAPACITY:.0%} of capacity)")
print(f"Months above capacity:      {(monthly_output > MONTHLY_CAPACITY).sum()}")

Monthly capacity (approx.): 130,000 pieces
Average monthly output:     174,103 pieces (134% of capacity)
Busiest month:              2024-07  ->  251,053 pieces (193% of capacity)
Months above capacity:      32


In [ ]:
# ============================================================
# 5.11 Memory, column order, final review, save
# ============================================================
for col in ["Product_ID", "Line_ID", "Primary_Bottleneck"]:
    prodn[col] = prodn[col].astype("category")

column_order = ["Production_Order_ID", "Product_ID", "Line_ID",
                "Start_Date", "Due_Date", "Completion_Date", "Is_Open",
                "Planned_Days", "Lead_Time_Days", "Delay_Days",
                "Planned_Qty", "Produced_Qty", "Good_Qty", "Rejected_Qty",
                "Fill_Rate", "Reject_Rate",
                "Efficiency_Pct", "Primary_Bottleneck", "Overtime_Hours"]
# Keep any extra columns we didn't know about at the end (don't lose them silently)
extra_cols = [c for c in prodn.columns if c not in column_order]
prodn = prodn[column_order + extra_cols]
if extra_cols:
    print("Extra columns kept at the end:", extra_cols)

print("Shape: before", raw["production"].shape, "-> after", prodn.shape)
print("\nColumns & types:")
for col in prodn.columns:
    print(f"  {col:<22} {str(prodn[col].dtype)}")

remaining_nulls = prodn.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nRemaining missing values (open orders have no completion date — expected):")
print(dict(remaining_nulls) if len(remaining_nulls) else "None")

prodn.to_parquet(CLEAN_DIR / "production_orders.parquet", index=False)
prodn.to_csv(CLEAN_DIR / "production_orders.csv", index=False, encoding="utf-8-sig")
print("\nSaved: production_orders.parquet + production_orders.csv")

In [68]:
# ============================================================
# 5.11 Memory, column order, final review, save
# ============================================================
for col in ["Product_ID", "Line_ID", "Primary_Bottleneck"]:
    prodn[col] = prodn[col].astype("category")

column_order = ["Production_Order_ID", "Product_ID", "Line_ID",
                "Start_Date", "Due_Date", "Completion_Date", "Is_Open",
                "Planned_Days", "Lead_Time_Days", "Delay_Days",
                "Planned_Qty", "Produced_Qty", "Good_Qty", "Rejected_Qty",
                "Fill_Rate", "Reject_Rate",
                "Efficiency_Pct", "Primary_Bottleneck", "Overtime_Hours"]
# Keep any extra columns we didn't know about at the end (don't lose them silently)
extra_cols = [c for c in prodn.columns if c not in column_order]
prodn = prodn[column_order + extra_cols]
if extra_cols:
    print("Extra columns kept at the end:", extra_cols)

print("Shape: before", raw["production"].shape, "-> after", prodn.shape)
print("\nColumns & types:")
for col in prodn.columns:
    print(f"  {col:<22} {str(prodn[col].dtype)}")

remaining_nulls = prodn.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nRemaining missing values (open orders have no completion date — expected):")
print(dict(remaining_nulls) if len(remaining_nulls) else "None")

prodn.to_parquet(CLEAN_DIR / "production_orders.parquet", index=False)
prodn.to_csv(CLEAN_DIR / "production_orders.csv", index=False, encoding="utf-8-sig")
print("\nSaved: production_orders.parquet + production_orders.csv")

Shape: before (2500, 16) -> after (2500, 19)

Columns & types:
  Production_Order_ID    str
  Product_ID             category
  Line_ID                category
  Start_Date             datetime64[us]
  Due_Date               datetime64[us]
  Completion_Date        datetime64[us]
  Is_Open                bool
  Planned_Days           int64
  Lead_Time_Days         int64
  Delay_Days             int64
  Planned_Qty            int64
  Produced_Qty           int64
  Good_Qty               int64
  Rejected_Qty           int64
  Fill_Rate              float64
  Reject_Rate            float64
  Efficiency_Pct         float64
  Primary_Bottleneck     category
  Overtime_Hours         float64

Remaining missing values (open orders have no completion date — expected):
None

Saved: production_orders.parquet + production_orders.csv


###  Production_Orders — Cleaning Summary
| Check / Decision | Result / Reason |
|---|---|
| `Production_Order_ID` unique | _fill in_ |
| "None" in `Primary_Bottleneck` read as missing by pandas → `"No bottleneck"` | pandas treats the word "None" as NaN by default; it means "no bottleneck" here |
| Dropped `Product_Name` (+ `Data_Class`) | Normalization |
| Dates → datetime; open orders = no `Completion_Date` | _fill in: N open orders_ |
| Due ≥ Start, Completion ≥ Start | _fill in_ |
| `Delay_Days` = max(Completion − Due, 0) | _fill in_ |
| Good + Rejected = Produced | _fill in_ |
| Efficiency within 0–100 | _fill in_ |
| All Product_IDs exist in catalog | _fill in_ |
| Added `Is_Open`, `Lead_Time_Days`, `Planned_Days`, `Fill_Rate`, `Reject_Rate` | Direct calculations used by the bot and analysis |
| Capacity check vs. 5,000 pieces/day | _fill in: avg % of capacity, busiest month_ |


# 6. Customer_Service


In [70]:
# ============================================================
# 6.1 Inspect: shape, types, missing, duplicates, distinct values
# ============================================================
cs = raw["customer_service"].copy()
print(f"Shape: {cs.shape[0]:,} x {cs.shape[1]}")
print("Columns:", list(cs.columns))
print("Raw Open_Date examples:", cs["Open_Date"].head(3).tolist())

print("\nMissing values:\n", cs.isna().sum())
print("\nDuplicated Ticket_ID:", cs["Ticket_ID"].duplicated().sum())

for col in ["Channel", "Issue_Type", "Resolved", "CSAT_1_to_5", "Data_Class"]:
    print(f"\n===== {col} =====")
    print(cs[col].value_counts(dropna=False))

Shape: 6,500 x 9
Columns: ['Ticket_ID', 'Open_Date', 'Channel', 'Issue_Type', 'First_Response_Minutes', 'Resolution_Hours', 'Resolved', 'CSAT_1_to_5', 'Data_Class']
Raw Open_Date examples: ['2025-02-02', '2024-01-01', '2026-03-21']

Missing values:
 Ticket_ID                 0
Open_Date                 0
Channel                   0
Issue_Type                0
First_Response_Minutes    0
Resolution_Hours          0
Resolved                  0
CSAT_1_to_5               0
Data_Class                0
dtype: int64

Duplicated Ticket_ID: 0

===== Channel =====
Channel
Instagram    1666
Phone        1624
WhatsApp     1612
Website      1598
Name: count, dtype: int64

===== Issue_Type =====
Issue_Type
Size guidance               1347
Order status                1273
General product question    1082
Exchange request             692
Product quality              668
Wholesale inquiry            599
Return request               526
Payment                      313
Name: count, dtype: int64

===== R

In [71]:
# ============================================================
# 6.2 Strip spaces + drop constant column
# ============================================================
for col in get_text_cols(cs):
    cs[col] = cs[col].str.strip()

print("Data_Class distinct values:", cs["Data_Class"].nunique())
cs = cs.drop(columns=["Data_Class"], errors="ignore")

print("Remaining columns:", list(cs.columns))

Data_Class distinct values: 1
Remaining columns: ['Ticket_ID', 'Open_Date', 'Channel', 'Issue_Type', 'First_Response_Minutes', 'Resolution_Hours', 'Resolved', 'CSAT_1_to_5']


In [72]:
# ============================================================
# 6.5 Open_Date: text -> date
# ============================================================
cs["Open_Date"] = parse_dates(raw["customer_service"]["Open_Date"], name="Open_Date")

print(f"\nRange: {cs['Open_Date'].min().date()} -> {cs['Open_Date'].max().date()}")

today = pd.Timestamp.today().normalize()
print("Tickets dated in the future:", (cs["Open_Date"] > today).sum())

print("\nTickets per year:")
print(cs["Open_Date"].dt.year.value_counts().sort_index())

[Open_Date] format used: %Y-%m-%d  ->  converted 6,500 of 6,500  (failed: 0)

Range: 2023-10-01 -> 2026-09-30
Tickets dated in the future: 0

Tickets per year:
Open_Date
2023     567
2024    2145
2025    2153
2026    1635
Name: count, dtype: int64


In [73]:
# ============================================================
# 6.6 Resolved: "Yes"/"No" -> boolean
# ============================================================
resolved_map = {"Yes": True, "No": False}
cs["Is_Resolved"] = cs["Resolved"].map(resolved_map)

# Values not in the dictionary become missing -> must be 0
unmapped = cs["Is_Resolved"].isna() & cs["Resolved"].notna()
print("Unmapped Resolved values:", unmapped.sum())
if unmapped.any():
    print("Values:", cs.loc[unmapped, "Resolved"].unique())

cs["Is_Resolved"] = cs["Is_Resolved"].astype("boolean")
cs = cs.drop(columns=["Resolved"], errors="ignore")

print(f"\nResolved tickets:   {cs['Is_Resolved'].sum():,}  ({cs['Is_Resolved'].mean():.1%})")
print(f"Unresolved tickets: {(~cs['Is_Resolved']).sum():,}")

Unmapped Resolved values: 0

Resolved tickets:   6,180  (95.1%)
Unresolved tickets: 320


In [74]:
# ============================================================
# 6.7 Numeric sanity and time logic
# ============================================================
num_cols = ["First_Response_Minutes", "Resolution_Hours", "CSAT_1_to_5"]
for col in num_cols:
    cs[col] = pd.to_numeric(cs[col], errors="coerce")

# 1) No negative times or ratings
negatives = (cs[num_cols] < 0).sum()
print("Negative values:", dict(negatives[negatives > 0]) if negatives.any() else "None")

# 2) CSAT must be between 1 and 5
bad_csat = cs["CSAT_1_to_5"].notna() & ~cs["CSAT_1_to_5"].between(1, 5)
print("CSAT outside 1–5:", bad_csat.sum())

# 3) The first reply must come BEFORE the issue is fully resolved
#    First_Response is in minutes, Resolution in hours -> convert to the same unit
response_hours = cs["First_Response_Minutes"] / 60
reply_after_resolution = cs["Is_Resolved"] & (response_hours > cs["Resolution_Hours"])
print("First reply came after resolution:", reply_after_resolution.sum())

# 4) Unresolved tickets: do they still have a resolution time?
unresolved = ~cs["Is_Resolved"]
print("\nUnresolved tickets WITH a Resolution_Hours value:",
      (unresolved & cs["Resolution_Hours"].notna()).sum())

print("\nSummary:")
print(cs[num_cols].describe().T.round(1))

Negative values: None
CSAT outside 1–5: 0
First reply came after resolution: 5

Unresolved tickets WITH a Resolution_Hours value: 320

Summary:
                         count  mean   std  min  25%   50%   75%    max
First_Response_Minutes  6500.0  11.5   9.1  1.0  5.6   8.9  14.5  109.3
Resolution_Hours        6500.0  17.4  14.0  0.2  8.0  15.8  23.7   95.9
CSAT_1_to_5             6500.0   4.0   1.1  1.0  3.0   4.0   5.0    5.0


In [75]:
# ============================================================
# 6.8 Evidence for the bot (printed now — final numbers go in the analysis step)
# ============================================================
# 1) What do customers ask about?
print("========== Tickets by issue type (% of all) ==========")
print((cs["Issue_Type"].value_counts(normalize=True) * 100).round(1))

# 2) Where do they contact us?
print("\n========== Tickets by channel (% of all) ==========")
print((cs["Channel"].value_counts(normalize=True) * 100).round(1))

# 3) Response time and satisfaction per issue type
print("\n========== Avg first response (min) and CSAT by issue type ==========")
print(cs.groupby("Issue_Type")[["First_Response_Minutes", "CSAT_1_to_5"]].mean().round(1))

# 4) Does waiting longer lead to lower satisfaction?
#    pd.cut splits a continuous number into labeled ranges ("buckets")
wait_bucket = pd.cut(cs["First_Response_Minutes"],
                     bins=[0, 5, 15, 30, 60, float("inf")],
                     labels=["0–5", "5–15", "15–30", "30–60", "60+"])
print("\n========== CSAT by first-response wait (minutes) ==========")
print(cs.groupby(wait_bucket, observed=True)["CSAT_1_to_5"].agg(["count", "mean"]).round(2))

========== Tickets by issue type (% of all) ==========
Issue_Type
Size guidance               20.7
Order status                19.6
General product question    16.6
Exchange request            10.6
Product quality             10.3
Wholesale inquiry            9.2
Return request               8.1
Payment                      4.8
Name: proportion, dtype: float64

========== Tickets by channel (% of all) ==========
Channel
Instagram    25.6
Phone        25.0
WhatsApp     24.8
Website      24.6
Name: proportion, dtype: float64

========== Avg first response (min) and CSAT by issue type ==========
                          First_Response_Minutes  CSAT_1_to_5
Issue_Type                                                   
Exchange request                            12.0          4.0
General product question                    11.5          4.0
Order status                                11.2          4.0
Payment                                     11.1          4.0
Product quality             

In [81]:
# ============================================================
# 6.9 Types, final review, save
# ============================================================
for col in ["Channel", "Issue_Type"]:
    cs[col] = cs[col].astype("category")

column_order = ["Ticket_ID", "Open_Date", "Channel", "Issue_Type",
                "First_Response_Minutes", "Resolution_Hours", "Is_Resolved", "CSAT_1_to_5"]
extra_cols = [c for c in cs.columns if c not in column_order]   # keep unknown columns
cs = cs[column_order + extra_cols]
if extra_cols:
    print("Extra columns kept at the end:", extra_cols)

print("Shape: before", raw["customer_service"].shape, "-> after", cs.shape)
print("\nColumns & types:")
for col in cs.columns:
    print(f"  {col:<24} {str(cs[col].dtype)}")

remaining_nulls = cs.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nRemaining missing values:", dict(remaining_nulls) if len(remaining_nulls) else "None")

cs.to_parquet(CLEAN_DIR / "customer_service.parquet", index=False)
cs.to_csv(CLEAN_DIR / "customer_service.csv", index=False, encoding="utf-8-sig")
print("\nSaved: customer_service.parquet + customer_service.csv")

Shape: before (6500, 9) -> after (6500, 8)

Columns & types:
  Ticket_ID                str
  Open_Date                datetime64[us]
  Channel                  category
  Issue_Type               category
  First_Response_Minutes   float64
  Resolution_Hours         float64
  Is_Resolved              boolean
  CSAT_1_to_5              int64

Remaining missing values: None

Saved: customer_service.parquet + customer_service.csv


| `Ticket_ID` unique | 0 duplicates |
| `Open_Date` → datetime | 6,500 tickets, 2023-10 → 2026-09 |
| `Is_Resolved` | 95.1% resolved (320 unresolved) |
| No negatives; CSAT 1–5 | Passed |
| First reply after resolution | 5 tickets (0.08%) — documented, not changed |
| Unresolved tickets with Resolution_Hours | All 320 → field likely means "time until closed"; analysis uses resolved tickets only |

**Findings:** CSAT ≈ 4.0 regardless of wait time or issue type → data does NOT support "faster reply = higher satisfaction".
Bot value argument = workload (~2,150 tickets/yr, 11.5 min avg first reply), 24/7 availability, and a dedicated "Wholesale inquiry" ticket type. Channels are evenly split (~25% each) → one backend serving all channels.


# 7. Group_Operations


In [76]:
# ============================================================
# 7.1 Working copy + first look
# ============================================================
gops = raw["group_ops"].copy()

print(f"Shape: {gops.shape[0]:,} rows x {gops.shape[1]} cols")
print("\nAll columns:", list(gops.columns))
print("\nRaw Month examples:", gops["Month"].head(3).tolist())
gops.head(10)

Shape: 180 rows x 7 cols

All columns: ['Month', 'Business_Unit', 'Primary_KPI', 'Volume', 'Operational_Efficiency_Pct', 'Open_Issues_Count', 'Data_Class']

Raw Month examples: ['2023-10-01', '2023-10-01', '2023-10-01']


,Month,Business_Unit,Primary_KPI,Volume,Operational_Efficiency_Pct,Open_Issues_Count,Data_Class
0,2023-10-01,LEO DESIGN,Designs_Completed,48,85.1,2,Modeled monthly operational summary using public busines...
1,2023-10-01,LEO TEX,Fabric_Meters,129278,91.5,6,Modeled monthly operational summary using public busines...
2,2023-10-01,LEO PRODUCTION HUB,Garments_Produced,78490,88.5,9,Modeled monthly operational summary using public busines...
3,2023-10-01,LEO PRINT,Print_Units,70454,86.7,1,Modeled monthly operational summary using public busines...
4,2023-10-01,LEO MODA,Sales_Units,36484,87.6,1,Modeled monthly operational summary using public busines...
5,2023-11-01,LEO DESIGN,Designs_Completed,72,77.8,0,Modeled monthly operational summary using public busines...
6,2023-11-01,LEO TEX,Fabric_Meters,153091,86.4,0,Modeled monthly operational summary using public busines...
7,2023-11-01,LEO PRODUCTION HUB,Garments_Produced,39649,83.5,5,Modeled monthly operational summary using public busines...
8,2023-11-01,LEO PRINT,Print_Units,152522,84.1,7,Modeled monthly operational summary using public busines...
9,2023-11-01,LEO MODA,Sales_Units,40007,90.7,0,Modeled monthly operational summary using public busines...


In [77]:
# ============================================================
# 7.2 Types, missing values, composite-key duplicates
# ============================================================
gops.info()

print("\nMissing values per column:")
print(gops.isna().sum())

KEY = ["Month", "Business_Unit"]
print("\nRows sharing the same (Month, Business_Unit):", gops.duplicated(subset=KEY).sum())
print("Fully duplicated rows:", gops.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Month                       180 non-null    str    
 1   Business_Unit               180 non-null    str    
 2   Primary_KPI                 180 non-null    str    
 3   Volume                      180 non-null    int64  
 4   Operational_Efficiency_Pct  180 non-null    float64
 5   Open_Issues_Count           180 non-null    int64  
 6   Data_Class                  180 non-null    str    
dtypes: float64(1), int64(2), str(4)
memory usage: 28.6 KB

Missing values per column:
Month                         0
Business_Unit                 0
Primary_KPI                   0
Volume                        0
Operational_Efficiency_Pct    0
Open_Issues_Count             0
Data_Class                    0
dtype: int64

Rows sharing the same (Month, Business_Unit): 0
Fully duplicated rows

In [78]:
# ============================================================
# 7.3 Distinct values + one KPI per business unit
# ============================================================
for col in ["Business_Unit", "Primary_KPI", "Data_Class"]:
    print(f"\n========== {col} ({gops[col].nunique()} distinct) ==========")
    print(gops[col].value_counts(dropna=False))

# Each unit must ALWAYS be measured by the same KPI,
# otherwise its Volume can't be compared month to month
kpis_per_unit = gops.groupby("Business_Unit")["Primary_KPI"].nunique()
print("\n========== KPIs per business unit (must all be 1) ==========")
print(kpis_per_unit)

print("\n========== Unit -> KPI ==========")
print(gops[["Business_Unit", "Primary_KPI"]].drop_duplicates().to_string(index=False))


========== Business_Unit (5 distinct) ==========
Business_Unit
LEO DESIGN            36
LEO TEX               36
LEO PRODUCTION HUB    36
LEO PRINT             36
LEO MODA              36
Name: count, dtype: int64

========== Primary_KPI (5 distinct) ==========
Primary_KPI
Designs_Completed    36
Fabric_Meters        36
Garments_Produced    36
Print_Units          36
Sales_Units          36
Name: count, dtype: int64

========== Data_Class (1 distinct) ==========
Data_Class
Modeled monthly operational summary using public business-unit structure    180
Name: count, dtype: int64

========== KPIs per business unit (must all be 1) ==========
Business_Unit
LEO DESIGN            1
LEO MODA              1
LEO PRINT             1
LEO PRODUCTION HUB    1
LEO TEX               1
Name: Primary_KPI, dtype: int64

========== Unit -> KPI ==========
     Business_Unit       Primary_KPI
        LEO DESIGN Designs_Completed
           LEO TEX     Fabric_Meters
LEO PRODUCTION HUB Garments_Produced
    

In [79]:
# ============================================================
# 7.4 Strip, drop constant column, date, numeric sanity
# ============================================================
for col in get_text_cols(gops):
    gops[col] = gops[col].str.strip()
gops = gops.drop(columns=["Data_Class"], errors="ignore")

# Month -> date (monthly data should always be on day 1)
gops["Month"] = parse_dates(raw["group_ops"]["Month"], name="Month")
print("Months not on day 1:", (gops["Month"].dt.day != 1).sum())

first, last = gops["Month"].min(), gops["Month"].max()
expected = len(pd.date_range(first, last, freq="MS"))
print(f"Range: {first.date()} -> {last.date()}  |  months: {gops['Month'].nunique()} (expected {expected})")

# Numeric sanity
num_cols = ["Volume", "Operational_Efficiency_Pct", "Open_Issues_Count"]
for col in num_cols:
    gops[col] = pd.to_numeric(gops[col], errors="coerce")

negatives = (gops[["Volume", "Open_Issues_Count"]] < 0).sum()
print("\nNegative values:", dict(negatives[negatives > 0]) if negatives.any() else "None")
print("Efficiency outside 0–100:",
      (~gops["Operational_Efficiency_Pct"].between(0, 100)).sum())

print("\nAverage per unit (Volume is NOT comparable across units — different KPIs):")
print(gops.groupby("Business_Unit")[num_cols].mean().round(1))

[Month] format used: %Y-%m-%d  ->  converted 180 of 180  (failed: 0)
Months not on day 1: 0
Range: 2023-10-01 -> 2026-09-01  |  months: 36 (expected 36)

Negative values: None
Efficiency outside 0–100: 0

Average per unit (Volume is NOT comparable across units — different KPIs):
                      Volume  Operational_Efficiency_Pct  Open_Issues_Count
Business_Unit                                                              
LEO DESIGN              65.5                        85.6                3.4
LEO MODA             46127.5                        89.6                3.1
LEO PRINT           130556.1                        86.0                3.9
LEO PRODUCTION HUB   90872.5                        85.7                6.1
LEO TEX             126047.0                        88.8                3.6


In [82]:
# ============================================================
# 7.5 Types, final review, save
# ============================================================
for col in ["Business_Unit", "Primary_KPI"]:
    gops[col] = gops[col].astype("category")

print("Shape: before", raw["group_ops"].shape, "-> after", gops.shape)
print("\nColumns & types:")
for col in gops.columns:
    print(f"  {col:<28} {str(gops[col].dtype)}")

remaining_nulls = gops.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nRemaining missing values:", dict(remaining_nulls) if len(remaining_nulls) else "None")

gops.to_parquet(CLEAN_DIR / "group_operations.parquet", index=False)
gops.to_csv(CLEAN_DIR / "group_operations.csv", index=False, encoding="utf-8-sig")
print("\nSaved: group_operations.parquet + group_operations.csv")

Shape: before (180, 7) -> after (180, 6)

Columns & types:
  Month                        datetime64[us]
  Business_Unit                category
  Primary_KPI                  category
  Volume                       int64
  Operational_Efficiency_Pct   float64
  Open_Issues_Count            int64

Remaining missing values: None

Saved: group_operations.parquet + group_operations.csv


###  Group_Operations — Cleaning Summary
| Composite key unique | 0 duplicates (5 units × 36 months = 180 rows) |
| One KPI per business unit | Passed |
| `Month` → datetime | 36 consecutive months, all on day 1 |
| No negatives; efficiency 0–100 | Passed (85.6%–89.6%) |
---
#  Notebook 01 — Output Summary
| File | Rows | Key facts |
|---|---|---|
| products.parquet | 59 | 50 Kids (Boys 25, Girls 21, Unisex 4) / 9 Women; Summer 45 / Autumn-Winter 14 |
| locations.parquet | 5 | 3 sales points (Sidi Bishr, Online, Wholesale Desk), all in Alexandria |
| customers.parquet | 30,000 | 1,651 wholesale traders (5.5%); 1,075 opted-in |
| inventory.parquet | 6,372 | 36 full months × 59 products × 3 locations; median cover 96 days; lost sales 0.9% |
| production_orders.parquet | 2,500 | all completed; lead time ~16 days; fill rate 91%; Sewing = main bottleneck |
| customer_service.parquet | _fill in_ | _fill in_ |
| group_operations.parquet | _fill in_ | _fill in_ |